# UH iGEM 2026 — Engineering _E. coli_ Nissle 1917 for α-Ketoglutarate Overproduction

## A Stage-by-Stage Computational Analysis Using Genome-Scale Metabolic Modeling

**Authors:** Austin Routt, UH iGEM 2026 Computational Team

**Model:** iHM1533 — _E. coli_ Nissle 1917 genome-scale metabolic model (van 't Hof et al., 2022)

**Software:** COBRApy 0.30.0, Python 3.8+, GLPK solver

---


## Table of Contents

- [Stage 0: Environment Setup & Model Loading](#stage-0-environment-setup--model-loading)
- [Stage 1: Wild-Type Characterization & Baseline Diagnostics](#stage-1-wild-type-characterization--baseline-diagnostics)
- [Stage 2: Tier Definitions & Production Envelopes](#stage-2-tier-definitions--production-envelopes)
- [Stage 3: Oxygen Sensitivity Sweep](#stage-3-oxygen-sensitivity-sweep)
- [Stage 4: Growth-Coupling Analysis (FVA)](#stage-4-growth-coupling-analysis-fva)
- [Stage 5: NADH Redox Balance](#stage-5-nadh-redox-balance)
- [Stage 6: Systematic Knockout Screen](#stage-6-systematic-knockout-screen)
- [Stage 7: Combinatorial Optimization](#stage-7-combinatorial-optimization-pairs--overexpression)
- [Stage 8: Transport & Export Analysis (KgtP)](#stage-8-transport--export-analysis-kgtp)
- [Stage 9: Heterologous Exporter Modeling](#stage-9-heterologous-exporter-modeling)
- [Stage 10: Citrate Synthase Feedback (Kinetic-FBA Hybrid)](#stage-10-citrate-synthase-feedback-kinetic-fba-hybrid)
- [Stage 11: Addiction System (PII/NtrC/thyA)](#stage-11-addiction-system-piintrcthya)
- [Stage 12: Glyoxylate Shunt Robustness](#stage-12-glyoxylate-shunt-robustness)
- [Stage 13: PYC vs PPC Anaplerotic Strategy](#stage-13-pyc-vs-ppc-anaplerotic-strategy)
- [Stage 14: Literature-Optimized Complete Strain](#stage-14-literature-optimized-complete-strain)
- [Stage 15: dFBA Plate Simulation](#stage-15-dfba-plate-simulation)
- [Stage 16: Final Publication Tables & Figures](#stage-16-final-publication-tables--figures)
- [Bibliography](#bibliography)

## Project Overview

The goal is to engineer _E. coli_ Nissle 1917 (EcN) to overproduce α-ketoglutarate (α-KG) and deliver it to _C. elegans_ to extend worm lifespan. Chin et al. (2014) demonstrated that 8 mM α-KG extends _C. elegans_ lifespan by approximately 50% through direct inhibition of the ATP synthase β-subunit (ATP-2), which reduces TOR signaling and increases autophagy — mimicking dietary restriction. Shahmirzadi et al. (2020) later showed that α-KG supplementation also compresses morbidity in aging mice, confirming its translational relevance.

EcN was chosen as the chassis because it is a clinically validated probiotic (Mutaflor), prescribed for ulcerative colitis in Europe (Kruis et al., 2004), and has established gut colonization properties with demonstrated epithelial barrier enhancement (Zyrek et al., 2007). Its genome-scale metabolic model, iHM1533, was published by van 't Hof et al. (2022) and contains 3,143 reactions, 2,261 metabolites, and 1,533 genes.

The experimental system uses _C. elegans_ grown on NGM agar plates (12 mL per 60 mm plate) supplemented with 0.4% glucose (~22 mM). A bacterial lawn is seeded and grown for 48 hours before worm transfer. The target is 8 mM α-KG accumulated in the plate volume. This is _not_ an industrial fermentation — it is a lawn on an agar plate.

The carbon source is glucose, not glycerol. This distinction is critical because glucose import via the PTS system consumes one PEP per glucose molecule, creating competition with PEP carboxylase (PPC) for the shared PEP pool. The published 44 g/L α-KG strain by Chiang et al. (2025) used glycerol, which bypasses PTS. Their _ppc_ overexpression strategy therefore does not directly translate to glucose-grown cells.




## Verified Reaction IDs in iHM1533

All reaction IDs used throughout this document have been verified against the loaded model. The complete reference table is:

|Role|Reaction ID|Key Stoichiometry|GPR|
|---|---|---|---|
|Biomass|`BIOMASS_EcN_iHM1533_core_59p80M`|(complex)|—|
|Glucose exchange|`EX_glc__D_e`|glc__D_e →|—|
|O₂ exchange|`EX_o2_e`|o2_e →|—|
|α-KG exchange|`EX_akg_e`|akg_e →|—|
|AKGDH (aerobic)|`AKGDH`|akg_c + coa_c + nad_c → co2_c + nadh_c + succoa_c|CIW80_21610 AND CIW80_18330 AND CIW80_21605|
|AKGDH (anaerobic)|`AKGDH2`|(analogous, different subunits)|CIW80_15445 AND CIW80_15450 AND CIW80_15455|
|D-Lactate DH|`LDH_D`|lac__D_c + nad_c ⇌ h_c + nadh_c + pyr_c|CIW80_25745 (ldhA)|
|Phosphotransacetylase|`PTAr`|accoa_c + pi_c ⇌ actp_c + coa_c|CIW80_05420 OR CIW80_06120|
|Glutamate DH|`GLUDy`|glu__L_c + h2o_c + nadp_c ⇌ akg_c + h_c + nadph_c + nh4_c|CIW80_01690 OR (CIW80_10545 AND CIW80_10540)|
|GOGAT|`GLUSy`|akg_c + gln__L_c + h_c + nadph_c → 2 glu__L_c + nadp_c|CIW80_10545 AND CIW80_10540|
|PEP carboxylase|`PPC`|co2_c + h2o_c + pep_c → h_c + oaa_c + pi_c|—|
|Citrate synthase|`CS`|accoa_c + h2o_c + oaa_c → cit_c + coa_c + h_c|—|
|Isocitrate lyase|`ICL`|icit_c → glx_c + succ_c|—|
|Malate synthase|`MALS`|accoa_c + glx_c + h2o_c → coa_c + h_c + mal__L_c|—|
|Malate DH|`MDH`|mal__L_c + nad_c ⇌ h_c + nadh_c + oaa_c|CIW80_10630|
|KgtP (α-KG transport)|`AKGt2rpp`|akg_p + h_p ⇌ akg_c + h_c|CIW80_06765|
|Outer membrane diffusion|`AKGtex`|akg_e ⇌ akg_p|(porins)|
|Thymidylate synthase|`TMDS`|dump_c + mlthf_c → dhf_c + dtmp_c|CIW80_07940|
|Thymidine exchange|`EX_thymd_e`|thymd_e →|—|
|Glutamine synthetase|`GLNS`|atp_c + glu__L_c + nh4_c → adp_c + gln__L_c + h_c + pi_c|—|

**Critical notes:**

- POXB (pyruvate oxidase) is **not present** in iHM1533. The wet lab should BLAST the EcN genome for a _poxB_ homolog.
- AKGDH and AKGDH2 share **zero genes** — they are true paralogs at different genomic loci. A single Δ_sucA_ at CIW80_21605 will not eliminate AKGDH2. Both must be knocked out.
- LDH_D2 (CIW80_04455) exists but is a quinone-linked lactate oxidase (utilization enzyme). It carries zero flux in Tier 2 production solutions. No additional knockout is needed.
- PTAr has an OR-rule GPR (CIW80_05420 OR CIW80_06120). Reaction-level knockout eliminates both genes. A single Δ_ptaA_ might leave the second gene functional. This must be stated in the Methods section of any publication.

---



## Stage 0: Environment Setup & Model Loading

**High-Level Summary**

This stage installs dependencies, loads the iHM1533 genome-scale metabolic model from its SBML file, validates all reaction IDs that will be used throughout the analysis, and configures the output directory. It also computes the wild-type aerobic oxygen baseline that defines the reference point for all downstream oxygen conditions. Every subsequent stage assumes this stage has been run.

**Justification**

Genome-scale metabolic models (GEMs) encode the complete network of biochemical reactions an organism can catalyze, along with gene-protein-reaction (GPR) associations that link genes to enzymatic functions (Orth et al., 2010). The iHM1533 model for _E. coli_ Nissle 1917 (EcN) contains 3,143 reactions, 2,261 metabolites, and 1,533 genes (van 't Hof et al., 2022). Loading it correctly and validating reaction IDs against the model is the absolute prerequisite for every downstream computation. A single mistyped reaction ID can produce silently incorrect results — the solver returns "optimal" but the intended knockout was never applied, because the wrong reaction was targeted.

COBRApy is the standard Python interface for constraint-based metabolic modeling. Standard FBA maximizes a single objective (here, biomass) subject to stoichiometric and bound constraints, but its solution is not unique — many flux distributions can achieve the same optimal growth rate. For later stages where we need unique, physiologically plausible flux distributions, we use pFBA (parsimonious FBA), which adds a secondary optimization that minimizes total network flux while preserving the optimal objective value. This selects the most enzymatically economical solution among all equivalent optima, eliminating thermodynamically infeasible loops (Lewis et al., 2010). For the baseline calculation in this stage, we use standard FBA because we only need the optimal growth rate and a representative O₂ uptake value to define the aerobic maximum.

An important subtlety: because FBA solutions are degenerate (multiple flux distributions can achieve the same growth rate), the exact O₂ uptake value returned by a single `model.optimize()` call is one point within the feasible range at the growth optimum. Different solvers, solver versions, or warm-start states can return slightly different O₂ values — all equally valid. This is acceptable because all downstream stages express oxygen conditions as fractions of whatever baseline is computed here, so the relative comparisons remain internally consistent. If absolute O₂ values matter for cross-study comparison, use FVA (Stage 4) to determine the full feasible range.

**Algorithm**

1. Import COBRApy, numpy, pandas, matplotlib, and pathlib.
2. Set matplotlib to a non-interactive backend (Agg) for script execution.
3. Create the output directory if it does not exist.
4. Load the iHM1533 SBML model using `read_sbml_model()`.
5. Print the model dimensions (reactions, metabolites, genes) as a sanity check.
6. Define a dictionary mapping human-readable role names to verified BiGG reaction IDs.
7. Iterate over this dictionary and confirm every ID exists in the model. Print a checkmark or warning for each. If any ID is missing, print candidate matches to aid correction, then abort.
8. Define the tier knockout lists that will be reused in all subsequent stages.
9. Determine the O₂ baseline within a `with model:` context manager (so all bound changes are automatically reversed afterward). Set medium conditions: glucose uptake at −10 mmol/gDW/hr, oxygen unconstrained, no exogenous α-KG. Maximize biomass. Record the growth rate and the absolute value of the O₂ uptake flux. This value defines 100% oxygen for all downstream analyses.

In [1]:
"""
Stage 0: Environment Setup & Model Loading
===========================================
Run this cell first. All subsequent stages assume these variables exist:
    model, BIOMASS_ID, GLC_EX_ID, O2_EX_ID, AKG_EX_ID,
    O2_BASELINE, WT_GROWTH, OUTPUT, TIERS

Design notes:
  - Standard FBA (not pFBA) is used for the baseline because we only
    need the optimal growth rate and a representative O₂ uptake, not a
    unique flux map. pFBA is used in later stages where full flux
    distributions matter (Stages 1, 5, 7+).
  - The `with model:` context manager ensures all bound changes inside
    the block are automatically reverted when the block exits. This
    prevents the baseline calculation from permanently altering the
    model that downstream stages depend on.
  - Because FBA is degenerate, the exact O₂ value can vary between
    solvers. All downstream stages use O2_BASELINE as a relative
    reference, so internal consistency is maintained regardless of
    the absolute value.
"""

# ─── Imports ─────────────────────────────────────────────────────────
# Core scientific stack
import warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd

# Plotting (imported here for use in all downstream stages)
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend; remove for Jupyter
import matplotlib.pyplot as plt

# Constraint-based modeling
import cobra
from cobra.io import read_sbml_model
from cobra.flux_analysis import pfba                     # Used in Stages 1, 5, 7+
from cobra.flux_analysis import flux_variability_analysis # Used in Stages 4, 8+

print(f"COBRApy version: {cobra.__version__}")

# ─── Output directory ────────────────────────────────────────────────
OUTPUT = Path("ecn_akg_outputs")
OUTPUT.mkdir(parents=True, exist_ok=True)

# ─── Load model ──────────────────────────────────────────────────────
# The iHM1533 model (van 't Hof et al., 2022, BMC Bioinformatics)
# is a genome-scale reconstruction of E. coli Nissle 1917 containing
# 3,143 reactions, 2,261 metabolites, and 1,533 genes.
# UPDATE THIS PATH to point to your local copy of iHM1533.xml.
MODEL_PATH = Path("iHM1533.xml")
model = read_sbml_model(str(MODEL_PATH))
print(f"Model: {model.id} | {len(model.reactions)} rxns | "
      f"{len(model.metabolites)} mets | {len(model.genes)} genes")

# ─── Master reaction ID registry ────────────────────────────────────
# Every reaction ID used anywhere in this analysis is registered here
# and validated against the loaded model BEFORE any computation runs.
# BiGG-standard IDs are used throughout (bigg.ucsd.edu).
BIOMASS_ID = "BIOMASS_EcN_iHM1533_core_59p80M"
GLC_EX_ID  = "EX_glc__D_e"
O2_EX_ID   = "EX_o2_e"
AKG_EX_ID  = "EX_akg_e"

RXN_IDS: Dict[str, str] = {
    "biomass":  BIOMASS_ID,
    "glc_ex":   GLC_EX_ID,
    "o2_ex":    O2_EX_ID,
    "akg_ex":   AKG_EX_ID,
    # TCA cycle targets (Tier 1 knockouts)
    "AKGDH":    "AKGDH",     # α-KG dehydrogenase (aerobic)
    "AKGDH2":   "AKGDH2",    # α-KG dehydrogenase (anaerobic paralog)
    # Fermentation targets (Tier 2 knockouts)
    "LDH_D":    "LDH_D",     # D-lactate dehydrogenase
    "PTAr":     "PTAr",      # Phosphotransacetylase
    # Nitrogen assimilation targets (Tier 2.5 / Tier 3)
    "GLUDy":    "GLUDy",     # Glutamate dehydrogenase (NADP)
    "GLUSy":    "GLUSy",     # Glutamate synthase / GOGAT
    # Overexpression & analysis targets
    "PPC":      "PPC",       # PEP carboxylase (anaplerosis)
    "CS":       "CS",        # Citrate synthase (TCA entry)
    "ICL":      "ICL",       # Isocitrate lyase (glyoxylate shunt)
    "MALS":     "MALS",      # Malate synthase (glyoxylate shunt)
    "MDH":      "MDH",       # Malate dehydrogenase
    "PPCK":     "PPCK",      # PEP carboxykinase
    "GLNS":     "GLNS",      # Glutamine synthetase (PII signaling)
    "TMDS":     "TMDS",      # Thymidylate synthase (addiction system)
}

# ─── Validate every reaction ID ─────────────────────────────────────
# This step catches typos and model-version mismatches BEFORE any
# downstream analysis runs. Without this check, a knockout of a
# nonexistent reaction silently does nothing — FBA returns "optimal"
# but the intended genetic modification was never applied.
print("\n=== Validating all reaction IDs ===")
all_valid = True
for role, rid in RXN_IDS.items():
    if rid in model.reactions:
        print(f"  ✓ {role:12s} → {rid}")
    else:
        print(f"  ✗ {role:12s} → {rid}  *** NOT FOUND ***")
        # Diagnostic: search for similarly named reactions to aid correction.
        # This saves time vs. manually scanning 3,143 reaction IDs.
        candidates = [r.id for r in model.reactions if rid.lower() in r.id.lower()]
        if candidates:
            print(f"      Possible matches: {candidates[:5]}")
        else:
            print(f"      No similar IDs found. Inspect the SBML file or "
                  f"use model.reactions.query('{rid}') interactively.")
        all_valid = False

assert all_valid, (
    "One or more reaction IDs were not found in the model. "
    "Inspect the ✗ lines above for suggestions, or use "
    "model.reactions.query('<partial_name>') to search interactively."
)

# ─── Tier definitions ────────────────────────────────────────────────
# The tiered knockout strategy is the core experimental design.
# Each tier adds knockouts to the previous tier. These lists are
# reused in every subsequent stage and must match the reaction IDs
# validated above.
#
#   WT:      No modifications (control)
#   Tier 1:  ΔsucA (AKGDH + AKGDH2) — block α-KG → succinyl-CoA
#   Tier 2:  + ΔldhA + Δpta — block lactate and acetate overflow
#   Tier 2.5: + ΔgdhA — block major α-KG consumer (glutamate DH)
#   Tier 3:  + ΔgltBD — block ALL glutamate biosynthesis (LETHAL)
TIERS: Dict[str, List[str]] = {
    "WT":       [],
    "Tier1":    ["AKGDH", "AKGDH2"],
    "Tier2":    ["AKGDH", "AKGDH2", "LDH_D", "PTAr"],
    "Tier2.5":  ["AKGDH", "AKGDH2", "LDH_D", "PTAr", "GLUDy"],
    "Tier3":    ["AKGDH", "AKGDH2", "LDH_D", "PTAr", "GLUDy", "GLUSy"],
}

# ─── O₂ baseline calculation ────────────────────────────────────────
# PURPOSE: Determine O₂ uptake at the WT growth optimum so all
# downstream oxygen conditions can be expressed as fractions of this
# value (e.g., "50% O₂" = 0.50 × O2_BASELINE).
#
# MEDIUM DEFINITION:
#   - Glucose: −10 mmol/gDW/hr (standard E. coli carbon constraint)
#   - O₂: unconstrained (lb = −1000) to find the aerobic maximum
#   - α-KG uptake: BLOCKED (lb = 0) so this reflects WT without
#     exogenous α-KG supplementation, which would inflate growth
#   - α-KG secretion: allowed (ub = 1000) to avoid artificially
#     constraining the model at this stage
#
# WHY `with model:`:
#   The context manager creates a snapshot of all model bounds.
#   When the block exits, bounds revert to their original values.
#   This prevents the baseline calculation from permanently altering
#   the model that all subsequent stages depend on.
#
# NOTE ON DEGENERACY:
#   At the growth optimum, multiple O₂ uptake values may be feasible
#   (alternate optima). The exact value returned depends on the LP
#   solver's tie-breaking. This is acceptable because all downstream
#   stages reference O2_BASELINE as a relative denominator, so the
#   fractional oxygen conditions (50%, 20%, etc.) remain internally
#   consistent. If you need the full feasible O₂ range, use FVA
#   (demonstrated in Stage 4).
with model:
    model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
    model.reactions.get_by_id(O2_EX_ID).lower_bound = -1000.0   # unconstrained O₂
    model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0      # no exogenous α-KG
    model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0   # allow secretion
    model.objective = BIOMASS_ID

    sol = model.optimize()

    # Validate solution before extracting values
    assert sol.status == "optimal", (
        f"WT optimization failed with status '{sol.status}'. "
        "Check medium bounds and model integrity."
    )

    # O₂ exchange flux is negative (uptake convention); take absolute value.
    # If for any reason the flux were positive (O₂ production), this would
    # be biologically nonsensical and should be flagged.
    o2_flux = sol.fluxes[O2_EX_ID]
    if o2_flux > 0:
        warnings.warn(
            f"O₂ flux is positive ({o2_flux:.4f}), indicating O₂ production. "
            "Check sign conventions and medium definition."
        )
    O2_BASELINE = abs(o2_flux)
    WT_GROWTH = sol.objective_value

    print(f"\nO₂ baseline (WT aerobic uptake): {O2_BASELINE:.4f} mmol/gDW/hr")
    print(f"WT growth rate: {WT_GROWTH:.4f} h⁻¹")

print(f"\nSetup complete. Output directory: {OUTPUT.resolve()}")

COBRApy version: 0.30.0
Model: iHM1533 | 3143 rxns | 2261 mets | 1533 genes

=== Validating all reaction IDs ===
  ✓ biomass      → BIOMASS_EcN_iHM1533_core_59p80M
  ✓ glc_ex       → EX_glc__D_e
  ✓ o2_ex        → EX_o2_e
  ✓ akg_ex       → EX_akg_e
  ✓ AKGDH        → AKGDH
  ✓ AKGDH2       → AKGDH2
  ✓ LDH_D        → LDH_D
  ✓ PTAr         → PTAr
  ✓ GLUDy        → GLUDy
  ✓ GLUSy        → GLUSy
  ✓ PPC          → PPC
  ✓ CS           → CS
  ✓ ICL          → ICL
  ✓ MALS         → MALS
  ✓ MDH          → MDH
  ✓ PPCK         → PPCK
  ✓ GLNS         → GLNS
  ✓ TMDS         → TMDS

O₂ baseline (WT aerobic uptake): 21.9718 mmol/gDW/hr
WT growth rate: 0.9668 h⁻¹

Setup complete. Output directory: C:\iGEM\ecn_akg_outputs


**Interpreting the Results**

When this cell runs successfully, you should see:

- **COBRApy version** — The exact version number is not critical as long as it is ≥ 0.26. Different versions may use slightly different solver defaults but should produce equivalent results for this model.
- **Model dimensions** — `3143 rxns | 2261 mets | 1533 genes` confirms the iHM1533 model loaded correctly. If these numbers differ, you may have a different model version.
- **Reaction ID validation** — A checkmark (✓) next to every reaction ID. If any show ✗, the ID does not exist under that name in the model. The diagnostic output will print candidate IDs to help you correct `RXN_IDS`. You can also search interactively in a Python session with `model.reactions.query("AKGDH")`.
- **O₂ baseline: 21.9718 mmol/gDW/hr** — This is the O₂ consumption rate at the WT aerobic growth optimum on 10 mmol/gDW/hr glucose with unlimited oxygen. All subsequent O₂ conditions (50%, 20%, etc.) are expressed as fractions of this value. Because FBA solutions can be degenerate (multiple equally optimal flux distributions exist), the exact O₂ value depends on your solver's tie-breaking behavior. A slightly different value (e.g., 19–22 mmol/gDW/hr) from a different solver or COBRApy version does not indicate an error — it means the solver chose a different point within the feasible range at the same growth optimum. What matters is that all downstream stages use whatever value `O2_BASELINE` holds, keeping the analysis internally consistent.
- **WT growth rate: 0.9668 h⁻¹** — The maximum predicted growth rate of wild-type EcN under the specified conditions. This value is uniquely determined (not degenerate) because biomass is the objective being maximized. This is a stoichiometric upper bound, not a kinetic prediction — actual growth rates in the lab will be lower due to regulatory and kinetic constraints not captured by FBA.

If the model file is not found, you will get a `FileNotFoundError`. Update `MODEL_PATH` to the correct location of your `iHM1533.xml` file. If the O₂ baseline is dramatically different (e.g., < 10 or > 40), verify that: (a) glucose uptake is set to −10.0, (b) α-KG uptake is blocked (lower_bound = 0.0), and (c) the SBML file's default exchange bounds for other nutrients have not been modified.

**ELI5 Summary**

Think of the genome-scale model as a massive blueprint of every chemical reaction EcN can perform — 3,143 reactions connected by 2,261 chemical substances. Before we can analyze anything, we need to load this blueprint into the computer and verify that all the room numbers (reaction IDs) we plan to visit actually exist in the building. If any room is missing, we print the names of nearby rooms to help you figure out what went wrong, then stop — it is better to fail loudly now than to quietly analyze the wrong room for the rest of the experiment.

The O₂ baseline is like measuring the top speed of a car under ideal highway conditions. We need to know what "full throttle" looks like so we can later enforce a "50% speed limit" and know it means exactly 50% of that measured maximum. We block α-KG uptake during this measurement (no exogenous supplements) so we're calibrating the car's natural performance, not a boosted version. And we do the whole measurement inside a temporary sandbox (the `with model:` block) so that when we're done calibrating, the car is returned to its original factory settings for the next experiment.

One subtlety: because there are many equally valid "routes" through the metabolic network at the same growth rate, different GPS systems (solvers) may report slightly different top speeds (O₂ values). This is fine — what matters is that every speed limit we set later is a fraction of whichever top speed we measured here, keeping everything consistent.

---

**References for this stage:**

- Lewis, N. E., Hixson, K. K., Conrad, T. M., et al. (2010). Omic data from evolved _E. coli_ are consistent with computed optimal growth from genome-scale models. _Molecular Systems Biology_, 6(1), 390.
- Orth, J. D., Thiele, I., & Palsson, B. Ø. (2010). What is flux balance analysis? _Nature Biotechnology_, 28(3), 245–248.
- van 't Hof, M. et al. (2022). iHM1533: A genome-scale metabolic model for _E. coli_ Nissle 1917. _BMC Bioinformatics_, 23, 454.

## Stage 1: Wild-Type Characterization & Baseline Diagnostics

**High-Level Summary**

This stage characterizes the wild-type (unmodified) metabolic state of EcN by running pFBA, identifying which reactions consume α-KG and produce succinyl-CoA, and running FVA to determine whether AKGDH flux is required or optional at the growth optimum. The diagnostic reveals a critical finding: knocking out AKGDH has almost no effect on growth because alternative succinyl-CoA routes exist that are equally growth-optimal, and the cell's demand for succinyl-CoA is negligibly small.

**Justification**

Before engineering any knockouts, we must understand where carbon flows in the wild-type network. Parsimonious FBA (pFBA) selects the flux distribution that minimizes total enzyme usage (the sum of absolute fluxes) while achieving maximum growth, providing the most physiologically plausible solution among many equivalent optima (Lewis et al., 2010). FVA (Flux Variability Analysis) then reveals the full range of flux each reaction can carry at the growth optimum — telling us whether a reaction is essential (minimum > 0), optional (minimum = 0, maximum > 0), or unused (maximum = 0) (Mahadevan & Schilling, 2003).

A key discovery from this diagnostic is that AKGDH has an FVA range of [0, 4.87] at the WT growth optimum. This means the reaction _can_ carry up to 4.87 mmol/gDW/hr but is never _required_ to carry any flux. In the pFBA solution, AKGDH carries 4.87 because it is the most flux-efficient (fewest total reactions) route to produce succinyl-CoA. However, because the FVA minimum is zero, we know an equally growth-optimal solution exists with AKGDH = 0. When pFBA is re-run on the knockout model (AKGDH blocked), the solver discovers that alternative solution — routing succinyl-CoA production through succinyl-CoA synthetase running in reverse — and achieves the same maximum growth rate. This is precisely why WT ≡ KO in naive growth comparisons: the knockout does not eliminate any growth-required reaction.

Why can the cell reroute so easily? The succinyl-CoA demand from the biomass equation is only 9.8 × 10⁻⁵ mmol/gDW per unit growth flux, as encoded in the iHM1533 biomass reaction (van 't Hof et al., 2022). At a growth rate of 0.97 h⁻¹, the actual demand is approximately 9.5 × 10⁻⁵ mmol/gDW/hr — a negligible quantity that can be met by succinyl-CoA synthetase running in reverse (succinate + CoA + ATP → succinyl-CoA), using succinate generated elsewhere in the TCA cycle. Note that the glyoxylate shunt is _not_ an alternative succinyl-CoA source — it bypasses both AKGDH and succinyl-CoA synthetase entirely, producing succinate and malate instead.

**Algorithm**

1. Set standard medium: glucose uptake = 10 mmol/gDW/hr (lower bound = −10), O₂ unconstrained (lower bound = −1000), allow α-KG secretion (upper bound = 1000), block α-KG uptake (lower bound = 0).
2. Run pFBA with biomass as the objective.
3. Extract the growth rate from `sol.fluxes[BIOMASS_ID]` — not from `sol.objective_value`. This distinction is critical: pFBA is a two-step optimization that first maximizes growth, then minimizes total flux. The `objective_value` attribute returns the minimized total flux sum (the second step's result), not the growth rate.
4. Report AKGDH, AKGDH2, and EX_akg_e fluxes.
5. Identify all reactions that net-consume cytosolic α-KG (akg_c) in this solution and rank by consumption rate.
6. Identify all reactions that net-produce succinyl-CoA (succoa_c) and rank them.
7. Extract the biomass succinyl-CoA stoichiometric coefficient and compute the actual demand at the observed growth rate.
8. Run FVA on AKGDH, AKGDH2, and EX_akg_e at 100% of the growth optimum.
9. Interpret the FVA ranges to determine essentiality and explain the weak knockout phenotype.

In [2]:
"""
Stage 1: Wild-Type Characterization & Baseline Diagnostics
===========================================================
Prerequisite: Stage 0 completed (model, all IDs, O2_BASELINE, OUTPUT defined).

PURPOSE:
  Map where carbon flows in wild-type EcN to understand WHY knocking
  out AKGDH has almost no effect on growth. The answer comes from two
  independent lines of evidence:
    1. pFBA shows the FLUX DISTRIBUTION at the growth optimum
    2. FVA shows the FEASIBLE RANGE of each reaction at the same optimum
  Together, they reveal that AKGDH is optional (FVA min = 0) even though
  pFBA uses it (pFBA flux = 4.87).

METHODS:
  pFBA = parsimonious FBA (Lewis et al., 2010): maximize growth, then
         minimize total flux. Gives a unique, physiologically plausible
         flux distribution.
  FVA  = Flux Variability Analysis (Mahadevan & Schilling, 2003): at a
         fixed growth optimum, find the min and max flux each reaction
         can carry across ALL alternate optimal solutions.
"""

print("=" * 70)
print("STAGE 1: WILD-TYPE CHARACTERIZATION")
print("=" * 70)

# ─── 1.1: WT pFBA at standard conditions ────────────────────────────
# We use the same medium as Stage 0 (glucose = -10, O₂ unconstrained,
# no exogenous α-KG). The `with model:` context ensures all bound
# changes revert when the block exits.
#
# WHY pFBA (not standard FBA):
#   Standard FBA returns ANY growth-optimal flux distribution, which
#   may include thermodynamically infeasible loops. pFBA adds a second
#   optimization step that minimizes total flux, eliminating such loops
#   and producing the most enzymatically economical solution.
with model:
    model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
    model.reactions.get_by_id(O2_EX_ID).lower_bound = -1000.0
    model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
    model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0
    model.objective = BIOMASS_ID

    sol_wt = pfba(model)

    # CRITICAL: pFBA is a two-step optimization:
    #   Step 1 — maximize biomass → records optimal growth rate µ*
    #   Step 2 — minimize Σ|v_i| subject to growth = µ*
    # Therefore sol.objective_value = minimized total flux (Step 2 result),
    # NOT the growth rate. Growth must be read from the flux vector.
    growth_rate = sol_wt.fluxes[BIOMASS_ID]
    total_flux  = sol_wt.objective_value

    print(f"\nWT pFBA results:")
    print(f"  Growth rate (from fluxes):    {growth_rate:.6f} h⁻¹")
    print(f"  Total flux sum (pFBA obj):    {total_flux:.2f}")
    print(f"  AKGDH flux:                   {sol_wt.fluxes['AKGDH']:.6f}")
    print(f"  AKGDH2 flux:                  {sol_wt.fluxes['AKGDH2']:.6f}")
    print(f"  α-KG exchange flux:           {sol_wt.fluxes[AKG_EX_ID]:.6f}")
    print(f"  Glucose uptake:               {sol_wt.fluxes[GLC_EX_ID]:.6f}")
    print(f"  O₂ uptake:                    {sol_wt.fluxes[O2_EX_ID]:.6f}")

# ─── 1.2: All reactions consuming akg_c ──────────────────────────────
# For each reaction involving akg_c, compute the NET contribution:
#   net = stoichiometric_coefficient × flux
# If net < 0, the reaction net-consumes akg_c in this solution.
#
# NOTE ON GLUDy SIGN CONVENTION:
#   iHM1533 encodes GLUDy's forward direction as:
#     glu_L_c + h2o_c + nadp_c → akg_c + h_c + nadph_c + nh4_c
#   (oxidative deamination: glutamate → α-KG)
#   A NEGATIVE flux means the reaction runs IN REVERSE:
#     akg_c + nadph_c + nh4_c → glu_L_c  (reductive amination)
#   This reverse direction is the biologically dominant one during
#   aerobic growth — the cell assimilates NH₄⁺ into glutamate using
#   α-KG as the carbon skeleton. So a negative GLUDy flux = α-KG
#   consumption for nitrogen assimilation. The sign reflects the
#   model's reaction convention, not a decrease in activity.
print("\n--- All reactions consuming akg_c in WT pFBA ---")
akg_c = model.metabolites.get_by_id("akg_c")
akg_consumers = []
for rxn in akg_c.reactions:
    coeff = rxn.get_coefficient(akg_c)
    flux = sol_wt.fluxes[rxn.id]
    consumption = -coeff * flux  # positive value = net consumption of akg_c
    if consumption > 0.001:
        akg_consumers.append({
            "reaction": rxn.id,
            "name": rxn.name,
            "akg_consumed": round(consumption, 4),
            "flux": round(flux, 4),
        })
akg_cons_df = pd.DataFrame(akg_consumers).sort_values(
    "akg_consumed", ascending=False
)
print(akg_cons_df.to_string(index=False))
akg_cons_df.to_csv(OUTPUT / "stage1_akg_consumers_wt.csv", index=False)

# ─── 1.3: All reactions producing succoa_c ───────────────────────────
# Same logic: net = coefficient × flux. Positive net = production.
# This identifies where succinyl-CoA comes from — and therefore what
# alternative routes exist when we knock out AKGDH.
print("\n--- All reactions producing succoa_c in WT pFBA ---")
succoa_c = model.metabolites.get_by_id("succoa_c")
succoa_producers = []
for rxn in succoa_c.reactions:
    coeff = rxn.get_coefficient(succoa_c)
    flux = sol_wt.fluxes[rxn.id]
    production = coeff * flux  # positive = net production of succoa_c
    if production > 0.001:
        succoa_producers.append({
            "reaction": rxn.id,
            "name": rxn.name,
            "succoa_produced": round(production, 4),
            "flux": round(flux, 4),
        })
succoa_df = pd.DataFrame(succoa_producers).sort_values(
    "succoa_produced", ascending=False
)
print(succoa_df.to_string(index=False))

# ─── 1.4: Biomass succoa_c coefficient ───────────────────────────────
# The biomass reaction specifies how much of each metabolite is needed
# per unit growth. A coefficient of -9.8e-05 means the cell consumes
# 9.8 × 10⁻⁵ mmol of succoa_c per gDW per h⁻¹ of growth rate.
# The ACTUAL DEMAND = |coefficient| × growth_rate.
# These are two distinct quantities — one is a model parameter, the
# other is the realized flux — and both are negligibly small.
bm_rxn = model.reactions.get_by_id(BIOMASS_ID)
for met, coeff in bm_rxn.metabolites.items():
    if "succoa" in met.id.lower():
        print(f"\nBiomass succoa_c stoichiometric coefficient: {coeff} "
              f"(mmol/gDW per unit growth flux)")
        print(f"  At growth = {growth_rate:.4f} h⁻¹, actual succoa demand = "
              f"{abs(coeff) * growth_rate:.6f} mmol/gDW/hr")
        break  # Only one succoa entry expected; prevent duplicate output

# ─── 1.5: FVA on key reactions at WT growth optimum ──────────────────
# FVA answers: "Across ALL flux distributions that achieve the same
# maximum growth rate, what is the minimum and maximum flux each
# reaction can carry?"
#
# fraction_of_optimum=1.0 means we lock growth at exactly 100% of
# the optimum. This gives the tightest bounds — the feasible range
# at EXACTLY maximum growth, not at reduced growth.
#
# KEY REACTIONS TO CHECK:
#   AKGDH  — Is the main α-KG drain required for growth?
#   AKGDH2 — Can the anaerobic paralog substitute?
#   EX_akg_e — Can the cell secrete α-KG without growth cost?
print("\n--- FVA at 100% WT growth optimum ---")
with model:
    model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
    model.reactions.get_by_id(O2_EX_ID).lower_bound = -1000.0
    model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
    model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0
    model.objective = BIOMASS_ID

    fva_result = flux_variability_analysis(
        model,
        reaction_list=["AKGDH", "AKGDH2", AKG_EX_ID],
        fraction_of_optimum=1.0,
    )
    print(fva_result)
    fva_result.to_csv(OUTPUT / "stage1_fva_wt.csv")

print("""
INTERPRETATION:

  AKGDH FVA range [0.0, 4.87]: The reaction CAN carry flux but is NOT
  REQUIRED to. pFBA assigns it 4.87 mmol/gDW/hr (the minimum-flux-norm
  solution uses AKGDH because it is the most direct succinyl-CoA route),
  but an equally growth-optimal solution with AKGDH = 0 exists. When
  pFBA is re-run on the knockout model, that alternative solution is
  found, achieving the same growth rate — hence WT ≡ KO in naive
  growth comparisons.

  AKGDH2 FVA range [0.0, 4.87]: The anaerobic paralog has the same
  feasible range, meaning it could stoichiometrically substitute for
  AKGDH at the growth optimum. pFBA sets it to zero under aerobic
  conditions because AKGDH alone satisfies parsimony, but AKGDH2
  remains a feasible alternative.

  EX_akg_e FVA range [0.0, ~0.0]: The maximum of ~1.7e-13 is numerical
  noise from the LP solver (any value below ~1e-9 should be treated as
  zero). At 100% growth optimum, the cell has NO stoichiometric latitude
  to secrete α-KG. Every mmol diverted to secretion would cost growth.

  This means knocking out AKGDH at the growth optimum changes nothing
  unless we FORCE α-KG production (by making secretion the objective).
  That is the basis for the production envelope analysis in Stage 2.
""")

STAGE 1: WILD-TYPE CHARACTERIZATION

WT pFBA results:
  Growth rate (from fluxes):    0.966795 h⁻¹
  Total flux sum (pFBA obj):    726.34
  AKGDH flux:                   4.865309
  AKGDH2 flux:                  0.000000
  α-KG exchange flux:           0.000000
  Glucose uptake:               -10.000000
  O₂ uptake:                    -19.814046

--- All reactions consuming akg_c in WT pFBA ---
reaction                           name  akg_consumed    flux
   GLUDy Glutamate dehydrogenase (NADP)        7.8304 -7.8304
   AKGDH   2-Oxogluterate dehydrogenase        4.8653  4.8653

--- All reactions producing succoa_c in WT pFBA ---
reaction                         name  succoa_produced   flux
   AKGDH 2-Oxogluterate dehydrogenase           4.8653 4.8653

Biomass succoa_c stoichiometric coefficient: -9.8e-05 (mmol/gDW per unit growth flux)
  At growth = 0.9668 h⁻¹, actual succoa demand = 0.000095 mmol/gDW/hr

--- FVA at 100% WT growth optimum ---
          minimum       maximum
AKGDH       

**Interpreting the Results**

The expected outputs and what they mean:

- **Growth rate = 0.9668 h⁻¹**: The maximum stoichiometric growth capacity of WT EcN on 10 mmol glucose/gDW/hr with unlimited O₂. In laboratory conditions, actual growth will be lower due to regulatory constraints, enzyme kinetics, and maintenance energy not captured by FBA. This is a stoichiometric upper bound, not a kinetic prediction.
    
- **O₂ uptake = −19.81 mmol/gDW/hr**: The negative sign indicates uptake (COBRApy convention: negative = entering the cell). Despite the O₂ lower bound being set to −1000 (effectively unlimited), the actual uptake is only 19.81. This is because the system is **carbon-limited, not oxygen-limited**: with only 10 mmol/gDW/hr of glucose entering the cell, the stoichiometry of oxidative phosphorylation determines how much O₂ can be reduced. Complete combustion of 10 mmol glucose would require ~60 mmol O₂ (6 mol O₂ per mol C₆H₁₂O₆), but a large fraction of carbon is diverted into biomass rather than being fully oxidized, reducing respiratory O₂ demand to ~20 mmol. Note also that this value (19.81, from pFBA) may differ from the O₂ baseline computed in Stage 0 (21.97, from standard FBA) — this is expected because FBA solutions are degenerate and pFBA's secondary flux-minimization step selects a different point in the feasible O₂ range. Both values correspond to the same optimal growth rate; they differ only in how the degenerate O₂ flux is resolved.
    
- **AKGDH flux = 4.8653**: In this pFBA solution, AKGDH is the sole succinyl-CoA producer, carrying 4.87 mmol/gDW/hr. This is the minimum-flux-norm solution — pFBA prefers AKGDH because it is the most direct route to succinyl-CoA. Other routes exist (verified: succinyl-CoA synthetase running in reverse provides 0.37 mmol/gDW/hr in the Tier 1 pFBA solution) but require additional reactions and therefore have a higher total-flux cost.
    
- **AKGDH2 flux = 0.0000**: The anaerobic paralog carries zero flux under aerobic conditions in this pFBA solution. However, the FVA maximum of 4.87 (identical to AKGDH's) shows it could fully substitute for AKGDH at the growth optimum. These are true paralogs at different genomic loci with zero shared genes (CIW80_21605/21610/18330 for AKGDH vs. CIW80_15445/15450/15455 for AKGDH2) — both must be knocked out to eliminate this reaction class entirely.
    
- **α-KG exchange = 0.0000**: WT does not secrete α-KG at the growth optimum. All produced α-KG is consumed internally. The FVA maximum of ~1.7 × 10⁻¹³ is numerical noise from the LP solver (below the solver's feasibility tolerance of ~10⁻⁹) and should be treated as exactly zero.
    
- **Top α-KG consumers — note on GLUDy flux sign**: GLUDy (glutamate dehydrogenase) appears with flux = −7.83 but akg_consumed = +7.83. The negative flux is not an error — it reflects the model's reaction convention. In iHM1533, GLUDy's forward direction is encoded as the oxidative deamination reaction (glutamate + NADP⁺ + H₂O → α-KG + NADPH + NH₄⁺). A negative flux means the reaction runs in _reverse_ — reductive amination (α-KG + NADPH + NH₄⁺ → glutamate) — which is the biologically dominant direction during aerobic growth on ammonium, when the cell assimilates nitrogen into amino acids using α-KG as the carbon skeleton. GLUDy at 7.83 mmol/gDW/hr is the **largest single consumer** of α-KG, substantially exceeding AKGDH at 4.87. This is why ΔgdhA (GLUDy knockout, Tier 2.5) is a valuable engineering target — it removes the biggest α-KG drain.
    
    Note: The model annotation "2-Oxogluterate dehydrogenase" contains a spelling error inherited from the SBML file; the correct name is **2-oxoglutarate dehydrogenase**. This does not affect computation.
    
- **Succinyl-CoA producers**: Only AKGDH appears above the 0.001 reporting threshold. Other reactions (e.g., succinyl-CoA synthetase, transaminases) carry zero or negligible succinyl-CoA production flux in this pFBA solution. However, FVA shows these reactions _can_ produce succinyl-CoA in alternative optimal solutions — this is why the knockout is viable.
    
- **Succinyl-CoA demand from biomass**: The stoichiometric coefficient is −9.8 × 10⁻⁵ mmol/gDW per unit growth flux (a model parameter). At growth = 0.9668 h⁻¹, the actual demand flux is 9.8 × 10⁻⁵ × 0.9668 ≈ 9.5 × 10⁻⁵ mmol/gDW/hr (a realized flux). Both are negligibly small. This explains why AKGDH knockout barely affects growth: the succinyl-CoA demand for biomass precursors is orders of magnitude smaller than AKGDH's capacity, and it can be met by succinyl-CoA synthetase running in reverse.
    
- **FVA ranges**: AKGDH [0, 4.87] and AKGDH2 [0, 4.87] — both are optional and interchangeable at the growth optimum. Neither is required. EX_akg_e [0, ~0] confirms that α-KG secretion is stoichiometrically incompatible with maximum growth.
    

A "good" result here is one where the diagnostics are internally consistent. If the AKGDH FVA minimum were > 0, the reaction would be essential and knockout would be lethal — contradicting the viable Tier 1 phenotype observed in Stage 2. The zero minimum confirms that alternative succinyl-CoA routes exist and that the knockout is viable.

**ELI5 Summary**

Imagine a factory (the cell) with a main assembly line (AKGDH) that converts a raw material (α-KG) into a specific component (succinyl-CoA). The factory also has a secondary line (succinyl-CoA synthetase running in reverse) that can make the same component from a different input. An efficiency auditor (pFBA) assigns work to the main line because it handles the job most directly. But if we shut down the main line (knockout), the factory barely notices — it only needs a tiny fraction of one component for its final product (biomass), and the secondary line easily covers that demand.

Meanwhile, the factory's biggest α-KG consumer is not the assembly line at all — it's the nitrogen-assimilation department (GLUDy), which converts α-KG into glutamate at nearly twice the rate of AKGDH. This tells us that to make α-KG accumulate and leave the factory, we need to close _both_ the assembly line door _and_ reduce the nitrogen department's appetite. That is the logic behind the tiered knockout strategy in Stage 2.

---

**References for this stage:**

- Lewis, N. E., Hixson, K. K., Conrad, T. M., et al. (2010). Omic data from evolved _E. coli_ are consistent with computed optimal growth from genome-scale models. _Molecular Systems Biology_, 6(1), 390.
- Mahadevan, R. & Schilling, C. H. (2003). The effects of alternate optimal solutions in constraint-based genome-scale metabolic models. _Metabolic Engineering_, 5(4), 264–276.
- van 't Hof, M. et al. (2022). iHM1533: A genome-scale metabolic model for _E. coli_ Nissle 1917. _BMC Bioinformatics_, 23, 454.

## Stage 2: Tier Definitions & Production Envelopes

**High-Level Summary**

This stage defines the tiered knockout strategy and computes the biomass–α-KG production envelope (Pareto frontier) for each tier at multiple oxygen levels. The production envelope reveals the fundamental trade-off between growth and α-KG secretion: how much α-KG can the cell secrete while maintaining at least X% of its maximum growth rate? This is the central analytical figure of the project — it drives every downstream engineering decision.

**Justification**

The production envelope is the standard FBA approach for visualizing the trade-off between growth and product secretion in metabolic engineering. It is constructed by iteratively maximizing the product flux (here, α-KG secretion via EX_akg_e) while constraining biomass to decreasing fractions of its maximum. This traces a discrete approximation of the Pareto frontier — the set of solutions where neither objective can be improved without worsening the other.

The tiered knockout strategy is informed by known metabolic engineering principles for α-KG accumulation:

- **Tier 1 (Δ_sucA_ × 2)** blocks both α-ketoglutarate dehydrogenase paralogs (AKGDH and AKGDH2), preventing α-KG conversion to succinyl-CoA. The "× 2" refers to the fact that AKGDH and AKGDH2 are true paralogs at different genomic loci sharing zero genes (CIW80_21605/21610/18330 vs CIW80_15445/15450/15455); a single Δ_sucA_ at CIW80_21605 will not eliminate AKGDH2. Both must be knocked out. The metabolic rationale for blocking this node is established by Li et al. (2006), who demonstrated that _sucA_ or _sucC_ knockout in _E. coli_ redirects flux away from the lower TCA cycle.
    
- **Tier 2** additionally blocks fermentation overflow routes: ΔldhA (LDH_D) eliminates D-lactate production; Δpta (PTAr) blocks the phosphotransacetylase step of the acetate pathway. Under microaerobic conditions where the ETC cannot reoxidize all NADH, these fermentation routes normally serve as electron sinks. Blocking them forces overflow carbon toward the TCA cycle and α-KG. Note: PTAr has an OR-rule GPR (CIW80_05420 OR CIW80_06120), so reaction-level knockout eliminates both gene products. A single Δ_ptaA_ in the wet lab might leave the second gene functional — this must be stated in Methods.
    
- **Tier 2.5** adds ΔgdhA (GLUDy), blocking the major α-KG consumer identified in Stage 1 (glutamate dehydrogenase, which consumed 7.83 mmol/gDW/hr in the WT pFBA solution). GOGAT (GLUSy) is preserved for essential glutamate synthesis, so the strain is not a glutamate auxotroph.
    
- **Tier 3** additionally blocks GOGAT (GLUSy), eliminating all glutamate biosynthesis from α-KG. This creates a complete glutamate auxotrophy and is expected to be non-viable on minimal medium. Tier 3 may be viable on rich medium containing peptone (see Stage 12 of the full analysis for NGM complex media simulation).
    

The key metric is **max α-KG at ≥80% growth**: the maximum α-KG secretion achievable while retaining at least 80% of the tier's maximum growth rate. This operating point represents a practical balance between production titer and strain fitness. The 80% threshold is a common choice in FBA strain design — it allows meaningful product secretion while maintaining enough growth for experimental viability (Chiang et al., 2025).

**Algorithm**

1. For each tier (WT, Tier 1, Tier 2, Tier 2.5, Tier 3): a. Apply the corresponding reaction knockouts (set lb = ub = 0). b. Set standard medium: glucose uptake = 10 mmol/gDW/hr, O₂ cap = O2_BASELINE × o2_frac, no exogenous α-KG. c. Maximize biomass → record µ_max. d. If µ_max < 0.01 h⁻¹, flag the tier as NON-VIABLE and skip. e. For each biomass fraction f in [1.0, 0.95, 0.90, 0.85, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10, 0.05, 0.0] (non-uniform spacing: finer near the 80% operating point, coarser below):
    - Set biomass lower bound = f × µ_max.
    - Maximize EX_akg_e.
    - Record the α-KG secretion rate, actual growth, glucose uptake, and yield.
2. Compile into a single DataFrame. Extract the ≥80% growth operating point for each tier × O₂ combination.
3. Generate production envelope figures.

In [3]:
"""
Stage 2: Tier Definitions & Production Envelopes
==================================================
Prerequisite: Stage 0 completed (model, all IDs, O2_BASELINE, TIERS, OUTPUT defined).

PURPOSE:
  Compute the biomass–α-KG Pareto frontier for each tier at multiple
  oxygen levels. This answers the central question: "How much α-KG can
  the cell secrete while maintaining at least X% of maximum growth?"

METHOD:
  Two-step optimization at each point on the envelope:
    Step 1: Maximize biomass → record µ_max for this tier + O₂ condition
    Step 2: Fix biomass ≥ f × µ_max, maximize α-KG secretion
  Sweep f from 1.0 (100% growth, no production) to 0.0 (zero growth,
  maximum theoretical production).

KEY DESIGN DECISIONS:
  - O₂ conditions are expressed as fractions of O2_BASELINE (from Stage 0).
    "50% O₂" means the O₂ uptake cap is set to 0.50 × O2_BASELINE, NOT
    50% dissolved oxygen in a bioreactor. Report the actual cap value
    alongside the percentage for reproducibility.
  - Tiers with µ_max < 0.01 h⁻¹ are flagged as NON-VIABLE and excluded
    from the envelope sweep entirely. This prevents the optimizer from
    returning biologically meaningless α-KG values at zero growth.
  - The TIERS dictionary is defined in Stage 0 and includes both AKGDH
    paralogs in Tier 1 (AKGDH and AKGDH2 share zero genes).
"""

print("=" * 70)
print("STAGE 2: TIER DEFINITIONS & PRODUCTION ENVELOPES")
print("=" * 70)

# Print key dependencies for reproducibility
print(f"  O2_BASELINE = {O2_BASELINE:.4f} mmol/gDW/hr (from Stage 0)")
print(f"  Tiers defined: {list(TIERS.keys())}")

# ─── 2.1: Core production envelope function ─────────────────────────
def run_production_envelope(model, knockouts, tier_name, o2_frac,
                            glc_uptake=10.0, biomass_fracs=None):
    """
    Compute the biomass–α-KG production envelope for a given set of
    knockouts and oxygen condition.

    Parameters
    ----------
    model         : cobra.Model (not permanently modified — uses context)
    knockouts     : list of str, reaction IDs to zero out
    tier_name     : str, label for output
    o2_frac       : float, fraction of O2_BASELINE (1.0 = fully aerobic)
    glc_uptake    : float, glucose uptake magnitude (mmol/gDW/hr)
    biomass_fracs : list of float, fractions of µ_max to sweep

    Returns
    -------
    pd.DataFrame with one row per biomass fraction, or empty if NON-VIABLE.
    """
    if biomass_fracs is None:
        # Non-uniform grid: finer resolution near the 80% operating point
        # (0.05 steps from 1.0 to 0.80), coarser below (0.10 steps).
        biomass_fracs = [1.0, 0.95, 0.90, 0.85, 0.80,
                         0.70, 0.60, 0.50, 0.40,
                         0.30, 0.20, 0.10, 0.05, 0.0]

    o2_cap = O2_BASELINE * o2_frac
    rows = []

    with model:
        # ── Set medium constraints ──
        # Glucose: fixed uptake rate (negative = into cell)
        model.reactions.get_by_id(GLC_EX_ID).lower_bound = -abs(glc_uptake)
        # O₂: capped at fraction of aerobic baseline
        model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(o2_cap)
        # α-KG: block uptake (lb=0), allow secretion (ub=1000)
        model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
        model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0

        # ── Apply knockouts (once; inherited by inner contexts) ──
        for ko_id in knockouts:
            model.reactions.get_by_id(ko_id).lower_bound = 0.0
            model.reactions.get_by_id(ko_id).upper_bound = 0.0

        # ── Step 1: Find maximum growth for this tier + O₂ ──
        model.objective = BIOMASS_ID
        model.objective_direction = "max"
        growth_sol = model.optimize()

        if growth_sol.status != "optimal":
            print(f"  {tier_name} at {o2_frac*100:.0f}% O₂: "
                  f"INFEASIBLE (status: {growth_sol.status})")
            return pd.DataFrame()

        mu_max = growth_sol.objective_value

        # Viability check: µ < 0.01 h⁻¹ is biologically non-viable
        if mu_max < 0.01:
            print(f"  {tier_name} at {o2_frac*100:.0f}% O₂: "
                  f"NON-VIABLE (µ={mu_max:.6f})")
            return pd.DataFrame()

        # ── Step 2: Sweep biomass fractions, maximize α-KG at each ──
        for frac in biomass_fracs:
            bm_floor = mu_max * frac

            # Inner context isolates the changing biomass floor while
            # inheriting medium and knockouts from the outer context.
            # No need to re-apply knockouts here — they persist.
            with model:
                model.reactions.get_by_id(BIOMASS_ID).lower_bound = bm_floor
                model.objective = AKG_EX_ID
                model.objective_direction = "max"
                akg_sol = model.optimize()

                if akg_sol.status == "optimal":
                    akg_flux = akg_sol.fluxes[AKG_EX_ID]
                    bm_actual = akg_sol.fluxes[BIOMASS_ID]
                    glc_flux = akg_sol.fluxes[GLC_EX_ID]
                else:
                    akg_flux = bm_actual = glc_flux = np.nan

                rows.append({
                    "tier": tier_name,
                    "knockouts": ",".join(knockouts) if knockouts else "none",
                    "o2_fraction": o2_frac,
                    "o2_cap_mmol": round(o2_cap, 2),
                    "biomass_fraction": frac,
                    "mu_max": mu_max,
                    "biomass_actual": bm_actual,
                    "akg_secretion": akg_flux,
                    "glucose_uptake": glc_flux,
                    "akg_yield": (
                        akg_flux / abs(glc_flux)
                        if np.isfinite(akg_flux) and abs(glc_flux) > 1e-9
                        else np.nan
                    ),
                })

    return pd.DataFrame(rows)


# ─── 2.2: Run envelopes for all tiers × O₂ conditions ───────────────
# O₂ conditions are fractions of O2_BASELINE, not absolute values.
# "50% O₂" means the uptake cap is 0.50 × 21.97 = 10.99 mmol/gDW/hr.
O2_CONDITIONS = [1.0, 0.50, 0.20]

all_envelopes = []
for tier_name, kos in TIERS.items():
    for o2_frac in O2_CONDITIONS:
        cap = O2_BASELINE * o2_frac
        print(f"  Running {tier_name} at {o2_frac*100:.0f}% O₂ "
              f"(cap = {cap:.2f} mmol/gDW/hr)...")
        df = run_production_envelope(model, kos, tier_name, o2_frac)
        if not df.empty:
            all_envelopes.append(df)

envelopes_df = pd.concat(all_envelopes, ignore_index=True)
envelopes_df.to_csv(OUTPUT / "stage2_all_envelopes.csv", index=False)

# ─── 2.3: Summary table — max α-KG at ≥80% growth ──────────────────
# For each tier × O₂, find the highest α-KG secretion among all
# biomass fractions ≥ 0.80. This is the "operating point" that
# balances production against strain fitness.
print("\n=== KEY RESULT: Max α-KG at ≥80% growth ===")
summary_rows = []
for tier_name in TIERS:
    for o2_frac in O2_CONDITIONS:
        subset = envelopes_df[
            (envelopes_df["tier"] == tier_name) &
            (envelopes_df["o2_fraction"] == o2_frac) &
            (envelopes_df["biomass_fraction"] >= 0.80)
        ].dropna(subset=["akg_secretion"])

        if not subset.empty:
            best = subset.loc[subset["akg_secretion"].idxmax()]
            summary_rows.append({
                "tier": tier_name,
                "o2_pct": int(o2_frac * 100),
                "o2_cap_mmol": round(O2_BASELINE * o2_frac, 2),
                "max_akg_at_80pct": round(best["akg_secretion"], 4),
                "growth_at_that_point": round(best["biomass_actual"], 4),
                "mu_max": round(best["mu_max"], 4),
                "akg_yield": round(best["akg_yield"], 4)
                    if np.isfinite(best["akg_yield"]) else np.nan,
            })
        else:
            # Tier was NON-VIABLE — record explicitly
            summary_rows.append({
                "tier": tier_name,
                "o2_pct": int(o2_frac * 100),
                "o2_cap_mmol": round(O2_BASELINE * o2_frac, 2),
                "max_akg_at_80pct": "NON-VIABLE",
                "growth_at_that_point": np.nan,
                "mu_max": np.nan,
                "akg_yield": np.nan,
            })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(OUTPUT / "stage2_tier_summary.csv", index=False)

# ─── 2.4: Production envelope figure ────────────────────────────────
# Tier 3 is excluded: it is NON-VIABLE and has no data in envelopes_df.
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
colors = {
    "WT": "#888888", "Tier1": "#2196F3", "Tier2": "#FF9800",
    "Tier2.5": "#9C27B0",
}

for idx, o2_frac in enumerate(O2_CONDITIONS):
    ax = axes[idx]
    # Dynamically plot whichever tiers have data at this O₂ level
    for tier_name in envelopes_df["tier"].unique():
        sub = envelopes_df[
            (envelopes_df["tier"] == tier_name) &
            (envelopes_df["o2_fraction"] == o2_frac)
        ].dropna(subset=["biomass_actual", "akg_secretion"])
        sub = sub.sort_values("biomass_actual")
        if not sub.empty:
            ax.plot(sub["biomass_actual"], sub["akg_secretion"], "o-",
                    color=colors.get(tier_name, "#333"),
                    label=tier_name, linewidth=2, markersize=4)

    ax.set_xlabel("Growth rate (h⁻¹)", fontsize=11)
    ax.set_ylabel("Max α-KG secretion (mmol/gDW/hr)", fontsize=11)
    cap_val = O2_BASELINE * o2_frac
    ax.set_title(f"{int(o2_frac*100)}% O₂ (cap = {cap_val:.1f})",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)

fig.suptitle("Production Envelopes: Growth vs α-KG Secretion\n"
             "(Tier 3 omitted — NON-VIABLE at all O₂ conditions)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT / "fig_stage2_envelopes.png", dpi=300)
fig.savefig(OUTPUT / "fig_stage2_envelopes.svg")
plt.close(fig)
print("\n  Saved fig_stage2_envelopes (.png and .svg)")

STAGE 2: TIER DEFINITIONS & PRODUCTION ENVELOPES
  O2_BASELINE = 21.9718 mmol/gDW/hr (from Stage 0)
  Tiers defined: ['WT', 'Tier1', 'Tier2', 'Tier2.5', 'Tier3']
  Running WT at 100% O₂ (cap = 21.97 mmol/gDW/hr)...
  Running WT at 50% O₂ (cap = 10.99 mmol/gDW/hr)...
  Running WT at 20% O₂ (cap = 4.39 mmol/gDW/hr)...
  Running Tier1 at 100% O₂ (cap = 21.97 mmol/gDW/hr)...
  Running Tier1 at 50% O₂ (cap = 10.99 mmol/gDW/hr)...
  Running Tier1 at 20% O₂ (cap = 4.39 mmol/gDW/hr)...
  Running Tier2 at 100% O₂ (cap = 21.97 mmol/gDW/hr)...
  Running Tier2 at 50% O₂ (cap = 10.99 mmol/gDW/hr)...
  Running Tier2 at 20% O₂ (cap = 4.39 mmol/gDW/hr)...
  Running Tier2.5 at 100% O₂ (cap = 21.97 mmol/gDW/hr)...
  Running Tier2.5 at 50% O₂ (cap = 10.99 mmol/gDW/hr)...
  Running Tier2.5 at 20% O₂ (cap = 4.39 mmol/gDW/hr)...
  Running Tier3 at 100% O₂ (cap = 21.97 mmol/gDW/hr)...
  Tier3 at 100% O₂: NON-VIABLE (µ=0.000000)
  Running Tier3 at 50% O₂ (cap = 10.99 mmol/gDW/hr)...
  Tier3 at 50% O₂: NON-VIA

**Interpreting the Results**

The summary table shows α-KG secretion at the ≥80% growth operating point (mmol/gDW/hr):

|Tier|100% O₂|50% O₂|20% O₂|
|---|---|---|---|
|WT|2.91|4.88|2.53|
|Tier 1|3.01|4.88|2.53|
|Tier 2|3.01|**6.50**|3.38|
|Tier 2.5|3.04|**6.67**|3.36|
|Tier 3|NON-VIABLE|NON-VIABLE|NON-VIABLE|

Key patterns:

- **Tier 1 barely improves over WT** (3.01 vs 2.91 at 100% O₂; identical at 50% and 20%). Stage 1 explained why: AKGDH is optional at the growth optimum (FVA minimum = 0), so knocking it out does not force the cell to do anything differently when growth is the priority. The improvement only becomes meaningful when we also block the fermentation safety valves (Tier 2).
    
- **Tier 1 at 50% O₂ equals WT at 50% O₂** (both 4.88). This is not a data error — it confirms that under microaerobic conditions with the 80% growth constraint, the growth-constrained production optimum does not route flux through AKGDH regardless of whether it is present. The knockout is redundant at this operating point.
    
- **Tier 2 at 50% O₂ shows the largest jump** (6.50 vs WT's 4.88, a 33% increase). Blocking fermentation overflow (ΔldhA + Δpta) is more impactful than blocking AKGDH because it forces carbon that would otherwise be wasted as lactate and acetate to flow through the TCA cycle toward α-KG. This effect is strongest at intermediate O₂ where the ETC is partially saturated and the cell would normally rely on fermentation as an electron sink.
    
- **Tier 2.5 slightly outperforms Tier 2 at 50% O₂** (6.67 vs 6.50). The ΔgdhA knockout removes the single largest α-KG consumer (GLUDy, 7.83 mmol/gDW/hr in WT pFBA — see Stage 1). The margin is modest because GOGAT (GLUSy) remains active and partially compensates. Growth is modestly reduced (µ_max = 0.556 vs 0.608) because GDH also contributes to NADPH recycling and nitrogen assimilation efficiency.
    
- **Tier 3 is non-viable on minimal medium**: Blocking both GLUDy (gdhA) and GLUSy (gltBD) eliminates all model routes from α-KG to glutamate, which is required for biomass synthesis. Growth = 0 at all oxygen levels. Because the code excludes non-viable tiers before the envelope sweep, there are no Tier 3 rows in `envelopes_df` — no artifact values to worry about. Note: Tier 3 may be viable on rich medium containing peptone (which provides exogenous glutamate); this is tested in a later stage.
    
- **50% O₂ dominates 100% O₂ for production** across Tier 2 and Tier 2.5. Microaerobic conditions restrict the ETC's NADH reoxidation capacity, forcing the cell to find alternative electron sinks. With fermentation blocked (Tier 2), α-KG secretion becomes one of the few remaining carbon outlets. At 100% O₂, the cell can fully oxidize glucose via the respiratory chain, routing carbon through CO₂ instead of α-KG — production suffers. At 20% O₂, growth is so constrained that overall carbon throughput drops, limiting production from the supply side. The 50% condition is the Goldilocks zone.
    
- **"50% O₂" means 50% of O2_BASELINE** (= 0.50 × 21.97 = 10.99 mmol/gDW/hr uptake cap). This is a model constraint, not a dissolved oxygen setpoint. Report the actual O₂ uptake cap alongside the percentage for reproducibility. The `o2_cap_mmol` column in the summary table serves this purpose.
    
- **The exact numerical values depend on O2_BASELINE**, which is determined by Stage 0. If you use a different method to compute the baseline (e.g., pFBA instead of standard FBA), the O₂ cap at each percentage will differ, producing different production values. The relative patterns (Tier 2 > Tier 1 ≈ WT; 50% O₂ > 100% O₂ > 20% O₂) are robust across reasonable baseline values.
    

The production envelope figures show the discrete Pareto frontier as connected points. The ≥80% growth operating point corresponds to the highest α-KG value among points where biomass fraction ≥ 0.80. Everything to the left of that region represents further growth sacrifice for increased production.

**ELI5 Summary**

Imagine each tier as a progressively modified factory. Tier 1 closes the main α-KG processing line (AKGDH) — but the factory barely notices because it has a workaround. Tier 2 additionally seals the lactate waste chute and the acetate side door. Now when the factory has more NADH than its power plant (ETC) can handle, it can't dump waste through fermentation — it has to push more material through the α-KG export dock instead. Tier 2.5 goes further by restricting the nitrogen department's biggest α-KG intake (GDH), while leaving the back door (GOGAT) open so the department can still function. Tier 3 shuts down the nitrogen department entirely — no glutamate, no proteins, no growth. The factory closes.

The production envelope asks: "If we tell the factory it must maintain at least 80% of its normal output (growth), how much α-KG can it divert to the export dock?" At 50% oxygen, the factory's power plant runs at half capacity, which is exactly the right amount of pressure to maximize α-KG export: enough to keep the assembly lines running, but not enough to burn all the waste internally. Tier 2.5 diverts 6.67 units at this sweet spot — slightly more than Tier 2's 6.50 — because the partially restricted nitrogen department consumes less of the α-KG raw material.

---

**References for this stage:**

- Chiang, C.-J. et al. (2025). Metabolic Engineering of _E. coli_ for Overproduction of Alpha-Ketoglutarate Using Crude Glycerol. _J. Agric. Food Chem._ 73, 18346–18352.
- Li, M. et al. (2006). Effect of _sucA_ or _sucC_ gene knockout on the metabolism in _E. coli_ based on gene expressions, enzyme activities, intracellular metabolite concentrations and metabolic fluxes by ¹³C-labeling experiments. _Biochem. Eng. J._ 30, 286–296.
- Noh, M.H. et al. (2017). Production of 5-aminolevulinic acid from succinyl-CoA in _E. coli_. _Process Biochem._ 56, 135–141.
- van 't Hof, M. et al. (2022). iHM1533: A genome-scale metabolic model for _E. coli_ Nissle 1917. _BMC Bioinformatics_ 23, 454.


## Stage 3: Oxygen Sensitivity Sweep

**High-Level Summary**

This stage sweeps oxygen availability from 0% to 100% of the aerobic maximum and records both growth rate and maximum α-KG production at ≥80% growth for each viable tier. The result identifies 50% O₂ as the production optimum for Tier 2 (6.50 mmol/gDW/hr) and Tier 2.5 (6.67 mmol/gDW/hr), and reveals the characteristic hump-shaped relationship between oxygen and α-KG yield — a non-monotonic curve driven entirely by the mathematical interplay between redox constraints and growth-imposed carbon demand.

**Justification**

Oxygen availability strongly affects _E. coli_ central carbon metabolism. Under lower-oxygen conditions, the regulators ArcA and FNR reshape expression of key TCA and respiratory genes; under more aerobic conditions, the full TCA/ETC program is active and respiratory flux is high (Alexeeva et al., 2002). Even under aerobic conditions, _E. coli_ exhibits overflow metabolism — primarily acetate secretion — at growth rates above roughly 0.35–0.45 h⁻¹, because the TCA cycle and ETC cannot absorb all NADH generated at high glycolytic flux (Vemuri et al., 2006). Under anaerobic conditions, the cell relies on mixed-acid fermentation (formate, acetate, ethanol, succinate, lactate) to regenerate NAD⁺, producing less ATP but conserving carbon in reduced products (Unden & Bongaerts, 1997). Anaerobic _E. coli_ can still grow — all tiers show non-zero growth rates at 0% O₂ in the output.

For α-KG production, this balance matters because α-KG is a TCA cycle intermediate. The enzyme α-ketoglutarate dehydrogenase (AKGDH, encoded by _sucAB_) converts α-KG to succinyl-CoA, generating additional NADH that must be reoxidized by the ETC. Under O₂ limitation, ETC capacity is insufficient to handle the full NADH load from completing the TCA cycle past AKGDH. Carbon therefore accumulates at α-KG and is secreted.

This sweep is essential for experimental design because _C. elegans_ plates are incubated under normal atmospheric conditions (aerobic at the agar surface) but the worm gut is likely microaerobic. The sweep tells us which oxygen regime the strain is optimized for, and whether plate-phase and gut-phase conditions will produce meaningfully different α-KG outputs.

**Important note for FBA interpretation:** In living cells, microaerobic conditions cause α-KG to accumulate partly through active transcriptional repression of AKGDH and downstream TCA enzymes by ArcA. FBA does not model transcriptional regulation — it has no concept of enzymes "backing up." The hump-shaped production curve in FBA emerges purely from stoichiometric and constraint-based math:

- **At low O₂ (redox limit):** Synthesizing α-KG from glucose generates NADH. Without sufficient oxygen as the terminal electron acceptor, the model cannot balance redox — it is mathematically bottlenecked. Production is constrained by NADH disposal capacity.
- **At high O₂ (biomass burden):** The maximum growth rate µ_max is very high (0.9551 h⁻¹ for Tier 2 at 100% O₂). Because the code enforces biomass ≥ 80% of µ_max, an enormous amount of carbon must be sunk into biomass at high O₂, leaving little surplus for α-KG secretion.
- **At 50% O₂ (sweet spot):** There is enough oxygen to solve the redox problem, but µ_max is moderate enough (0.6084 h⁻¹) that the 80% growth floor does not consume all available carbon.

This distinction between biological mechanism and FBA constraint is critical for interpreting the results correctly and communicating them in a publication.

**One additional subtlety:** The 80% growth constraint is _condition-specific_ — it is recalculated as 80% of each oxygen level's own µ_max. This means the α-KG curve reflects a combined effect of oxygen availability _plus_ a changing biomass requirement, not a pure oxygen-only comparison. This is the standard approach for production envelope analysis, but it should be stated explicitly.

**Algorithm**

1. Define O₂ sweep points: [0, 5, 10, 15, 20, 30, 40, 50, 75, 100]% of the aerobic maximum O₂ uptake (O2_BASELINE from Stage 0).
2. For each viable tier (excluding Tier 3, which is non-viable — confirmed in Stage 2) and each O₂ fraction: a. Apply knockouts; set glucose and O₂ exchange bounds. b. Maximize biomass → record µ_max. c. With biomass ≥ 80% of µ_max, maximize α-KG → record `akg_at_80pct_growth`. This is the production achievable while maintaining robust growth. d. With no growth constraint (biomass lower bound = 0), maximize α-KG → record `akg_unconstrained`. This theoretical ceiling shows how much yield is sacrificed by the 80% growth requirement.
3. Plot a two-panel figure: growth rate (Panel A) and α-KG production (Panel B, both constrained and unconstrained) vs. O₂%.
4. Identify the optimal O₂ fraction for each tier based on constrained production.

In [5]:
"""
Stage 3: Oxygen Sensitivity Sweep
===================================
Prerequisite: Stage 0 completed (model, TIERS, O2_BASELINE, all IDs defined).

PURPOSE:
  Sweep O₂ from 0–100% of the aerobic maximum and measure the growth
  rate and α-KG production capacity at each level. This identifies the
  oxygen condition that maximizes α-KG output while maintaining viable
  growth — the "production-optimal O₂."

METHOD:
  At each O₂ level, a three-step optimization:
    Step 1: Maximize biomass → µ_max at this O₂
    Step 2: Fix biomass ≥ 80% of µ_max, maximize α-KG → constrained production
    Step 3: Fix biomass ≥ 0, maximize α-KG → unconstrained theoretical ceiling
  The constrained value (Step 2) is the practical operating point.
  The ceiling (Step 3) shows how much production is sacrificed for growth.

WHY THE HUMP SHAPE (FBA vs biology):
  In living cells, microaerobic conditions cause α-KG accumulation
  through transcriptional regulation (ArcA represses downstream TCA).
  FBA does NOT model regulation. The hump in FBA arises from math:
    - Low O₂ → redox bottleneck (can't reoxidize NADH from TCA flux)
    - High O₂ → biomass burden (80% of a high µ_max eats all carbon)
    - 50% O₂ → sweet spot (enough O₂ for redox, moderate µ_max)
  Both the biological and mathematical mechanisms converge on the same
  conclusion (microaerobic is optimal), but for different reasons.

NOTE ON THE 80% CONSTRAINT:
  The growth floor is condition-specific: 80% of each O₂ level's own
  µ_max. This means the production curve reflects BOTH oxygen effects
  and the changing biomass requirement. This is the standard approach,
  but readers should be aware it is not a pure oxygen-only comparison.
"""

print("=" * 70)
print("STAGE 3: OXYGEN SENSITIVITY SWEEP")
print("=" * 70)

# O₂ sweep points: fractions of O2_BASELINE (from Stage 0)
# "50% O₂" = 0.50 × O2_BASELINE, NOT 50% dissolved oxygen
O2_SWEEP = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.75, 1.0]

sweep_rows = []
for tier_name, kos in TIERS.items():
    if tier_name == "Tier3":
        # Tier 3 is non-viable at all O₂ levels (confirmed in Stage 2).
        # Sweeping it would return µ=0 everywhere. Skip.
        continue

    for o2_frac in O2_SWEEP:
        o2_cap = O2_BASELINE * o2_frac
        with model:
            # ── Set medium and knockouts ──
            model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
            model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(o2_cap)
            model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
            model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0
            for ko_id in kos:
                model.reactions.get_by_id(ko_id).lower_bound = 0.0
                model.reactions.get_by_id(ko_id).upper_bound = 0.0

            # ── Step 1: Maximize biomass ──
            model.objective = BIOMASS_ID
            model.objective_direction = "max"
            g_sol = model.optimize()
            mu_max = g_sol.objective_value if g_sol.status == "optimal" else 0.0

            # ── Step 2: Max α-KG at ≥80% growth (practical operating point) ──
            akg_80 = 0.0
            if mu_max > 1e-6:
                with model:
                    # Knockouts and medium inherited from outer context.
                    model.reactions.get_by_id(BIOMASS_ID).lower_bound = mu_max * 0.80
                    model.objective = AKG_EX_ID
                    model.objective_direction = "max"
                    a_sol = model.optimize()
                    akg_80 = (a_sol.fluxes[AKG_EX_ID]
                              if a_sol.status == "optimal" else 0.0)

            # ── Step 3: Max α-KG unconstrained (theoretical ceiling) ──
            # Shows maximum possible α-KG if growth is entirely sacrificed.
            # The gap between akg_80 and akg_unconstrained quantifies
            # how much production is "lost" to the 80% growth requirement.
            akg_unconstrained = 0.0
            with model:
                model.reactions.get_by_id(BIOMASS_ID).lower_bound = 0.0
                model.objective = AKG_EX_ID
                model.objective_direction = "max"
                u_sol = model.optimize()
                akg_unconstrained = (u_sol.fluxes[AKG_EX_ID]
                                     if u_sol.status == "optimal" else 0.0)

        sweep_rows.append({
            "tier": tier_name,
            "o2_frac": o2_frac,
            "o2_pct": int(o2_frac * 100),
            "o2_cap_mmol": round(o2_cap, 2),
            "mu_max": round(mu_max, 4),
            "akg_at_80pct_growth": round(akg_80, 4),
            "akg_unconstrained": round(akg_unconstrained, 4),
        })

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(OUTPUT / "stage3_o2_sweep.csv", index=False)

# ── Print pivot tables ──────────────────────────────────────────────
tier_order = [t for t in ["WT", "Tier1", "Tier2", "Tier2.5"]
              if t in sweep_df["tier"].values]

print("\nMax α-KG at ≥80% growth (rows=O₂%, cols=tier):")
pivot_akg = sweep_df.pivot(
    index="o2_pct", columns="tier", values="akg_at_80pct_growth"
)
print(pivot_akg[tier_order].round(3).to_string())

print("\nTheoretical ceiling — unconstrained (rows=O₂%, cols=tier):")
pivot_ceil = sweep_df.pivot(
    index="o2_pct", columns="tier", values="akg_unconstrained"
)
print(pivot_ceil[tier_order].round(3).to_string())

print("\nGrowth rate (rows=O₂%, cols=tier):")
pivot_mu = sweep_df.pivot(index="o2_pct", columns="tier", values="mu_max")
print(pivot_mu[tier_order].round(4).to_string())

# ── Figure ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
colors = {"WT": "#888888", "Tier1": "#2196F3",
          "Tier2": "#FF9800", "Tier2.5": "#9C27B0"}

for tier_name in tier_order:
    sub = sweep_df[sweep_df["tier"] == tier_name]

    # Panel A: growth rate
    ax1.plot(sub["o2_pct"], sub["mu_max"], "o-",
             color=colors[tier_name], label=tier_name, linewidth=2)

    # Panel B: α-KG production
    # Solid = constrained (≥80% growth); Dashed = unconstrained ceiling
    ax2.plot(sub["o2_pct"], sub["akg_at_80pct_growth"], "o-",
             color=colors[tier_name], label=f"{tier_name} (≥80% growth)",
             linewidth=2)
    ax2.plot(sub["o2_pct"], sub["akg_unconstrained"], "o--",
             color=colors[tier_name], label=f"{tier_name} (ceiling)",
             linewidth=1.5, alpha=0.5)

# Highlight 50% O₂ optimum — MUST be added BEFORE legend() call
# so the label appears in the legend
ax2.axvline(50, color="red", ls="--", alpha=0.4, label="50% O₂ optimum")

ax1.set_xlabel("O₂ (% of aerobic maximum)", fontsize=12)
ax1.set_ylabel("Growth rate (h⁻¹)", fontsize=12)
ax1.set_title("A. Growth rate vs oxygen", fontsize=13, fontweight="bold")
ax1.legend(fontsize=10)

ax2.set_xlabel("O₂ (% of aerobic maximum)", fontsize=12)
ax2.set_ylabel("Max α-KG (mmol/gDW/hr)", fontsize=12)
ax2.set_title("B. α-KG production capacity vs oxygen", fontsize=13, fontweight="bold")
ax2.legend(fontsize=8)

fig.tight_layout()
fig.savefig(OUTPUT / "fig_stage3_o2_sweep.png", dpi=300)
fig.savefig(OUTPUT / "fig_stage3_o2_sweep.svg")
plt.close(fig)
print("\n  Saved fig_stage3_o2_sweep (.png and .svg)")

# ── Identify optimal O₂ for each tier ────────────────────────────────
print("\nOptimal O₂ by tier (maximizing α-KG at ≥80% growth):")
for tier_name in tier_order:
    sub = sweep_df[sweep_df["tier"] == tier_name]
    best = sub.loc[sub["akg_at_80pct_growth"].idxmax()]
    ceiling = best["akg_unconstrained"]
    efficiency = (100 * best["akg_at_80pct_growth"] / ceiling
                  if ceiling > 0 else float("nan"))
    print(f"  {tier_name:8s}: {int(best['o2_pct'])}% O₂  |  "
          f"α-KG = {best['akg_at_80pct_growth']:.3f}  |  "
          f"ceiling = {ceiling:.3f}  |  "
          f"efficiency = {efficiency:.1f}%  |  "
          f"µ = {best['mu_max']:.4f} h⁻¹")

STAGE 3: OXYGEN SENSITIVITY SWEEP

Max α-KG at ≥80% growth (rows=O₂%, cols=tier):
tier       WT  Tier1  Tier2  Tier2.5
o2_pct                              
0       1.209  1.209  1.404    1.385
5       1.539  1.539  1.875    1.849
10      1.868  1.868  2.358    2.335
15      2.198  2.198  2.868    2.841
20      2.528  2.528  3.379    3.364
30      3.206  3.206  4.465    4.467
40      4.240  4.240  5.566    5.569
50      4.879  4.879  6.505    6.669
75      4.144  4.185  4.582    5.111
100     2.914  3.009  3.010    3.039

Theoretical ceiling — unconstrained (rows=O₂%, cols=tier):
tier        WT   Tier1   Tier2  Tier2.5
o2_pct                                 
0        5.445   5.445   5.445    5.445
5        6.737   6.737   6.737    6.737
10       8.030   8.030   8.030    8.030
15       9.322   9.322   9.322    9.322
20      10.290  10.290  10.290   10.237
30      11.511  11.511  11.511   11.236
40      12.695  12.695  12.695   12.235
50      12.762  12.762  12.762   12.597
75      12.762

**Interpreting the Results**

The oxygen sweep reveals a characteristic hump-shaped curve for α-KG production across all viable tiers, driven by a mathematical trade-off between redox capacity and biomass burden:

- **0% O₂ (anaerobic):** Production is low but non-zero (~1.2–1.4 mmol/gDW/hr across tiers). All tiers still grow anaerobically via mixed-acid fermentation — note the non-zero µ_max values (WT: 0.1887 h⁻¹, Tier 2: 0.1440 h⁻¹). Without ETC activity, NADH generated by TCA flux cannot be reoxidized through respiration, severely limiting how much α-KG the stoichiometry permits.
    
- **20–40% O₂:** Production climbs steeply. The ETC absorbs enough NADH to allow meaningful TCA flux up to α-KG, while the O₂ limitation still prevents the cell from efficiently running the full downstream cycle.
    
- **50% O₂ (the optimum):** Peak production for Tier 2 at **6.50 mmol/gDW/hr** (growth = 0.6084 h⁻¹) and Tier 2.5 at **6.67 mmol/gDW/hr** (growth = 0.5562 h⁻¹). At this oxygen level, two conditions converge favorably: (a) the ETC handles sufficient NADH to sustain TCA flux up to α-KG, but cannot handle the additional NADH from running past AKGDH, so carbon accumulates at α-KG; (b) the 80% growth floor is applied to a moderate µ_max (0.6084 for Tier 2), so the absolute biomass carbon demand is modest, leaving more carbon available for secretion.
    
- **75–100% O₂:** Production drops steeply (Tier 2 falls to **3.01** at 100% O₂). This is the _biomass burden_ effect: at 100% O₂, Tier 2's µ_max is 0.9551 h⁻¹, and 80% of that (0.764 h⁻¹) demands an enormous fraction of the incoming 10 mmol glucose just for growth. Very little carbon remains for α-KG secretion. Additionally, with abundant O₂, the ETC is no longer limiting and can reoxidize all NADH from the complete TCA cycle, removing the thermodynamic pressure to truncate at α-KG. Aerobic overflow metabolism (primarily acetate secretion) also competes for carbon at high growth rates (Vemuri et al., 2006).
    

**Growth rate curves** show a monotonic increase with O₂ for all tiers. This monotonic growth vs. non-monotonic production is the key design tension: **the O₂ level that maximizes biomass (100%) is not the O₂ level that maximizes α-KG yield (50%).**

**Tier comparison at 50% O₂:** Tier 2.5 slightly outperforms Tier 2 (6.67 vs 6.50) because the additional ΔgdhA knockout removes the largest α-KG consumer (GLUDy, identified in Stage 1 as consuming 7.83 mmol/gDW/hr in WT). The cost is a modestly lower growth rate (0.5562 vs 0.6084), since GDH also contributes to nitrogen assimilation efficiency.

**Tier 1 ≈ WT at all O₂ levels below 75%:** This confirms Stage 1's finding that AKGDH knockout alone is insufficient — the flux removed by knocking out AKGDH is not rate-limiting for α-KG accumulation. The fermentation knockouts in Tier 2 (ΔldhA + Δpta) are what create the meaningful production jump.

**The theoretical ceiling** (dashed lines in Panel B) shows the maximum α-KG achievable at each O₂ level if growth is entirely sacrificed (biomass floor = 0). The gap between the solid and dashed curves quantifies the production cost of the 80% growth requirement. At 50% O₂, this gap shows that maintaining 80% growth sacrifices roughly one-third of the theoretical maximum — a significant but acceptable cost for experimental viability.

**Relevance to the worm experiment:** The plate surface during the 48-hour lawn growth phase is fully aerobic. Based on this sweep, plate-phase α-KG production for Tier 2 is roughly 3.0 mmol/gDW/hr — approximately half of the 50% O₂ optimum. Once _C. elegans_ ingests the bacteria, the worm gut transitions them to a likely microaerobic environment closer to the 50% O₂ regime. This means the strain is predicted to produce α-KG more efficiently _inside the worm_ than _on the plate_ — an advantageous synergy for the delivery strategy.

**ELI5 Summary**

Think of oxygen like the electricity supply to a factory, and glucose as the raw materials. Making α-KG (a toy) requires electricity to run the assembly line.

At **0% power** (anaerobic), the factory barely has enough electricity to run the toy machines, so production is minimal — though the factory doesn't shut down entirely; workers can still do some manual assembly.

As power increases, you can run more toy machines and produce more toys. **But** management has a strict rule: _you must always dedicate 80% of your maximum possible factory expansion rate to building new factory wings before making toys._

At **100% power**, your potential expansion rate is massive! To meet the 80% quota, almost all your raw materials are consumed just building the factory itself, leaving barely anything for toys. Additionally, with full power the factory can now process materials past the toy station entirely, so less piles up there.

**50% power** is the sweet spot: you have enough electricity to run the toy line efficiently, but your factory expansion quota is low enough that plenty of raw materials are left over for the toys. This is why the "medium heat" setting produces the most toys — not because of any magical property of 50%, but because it balances two competing demands.

---

**References for this stage:**

- Alexeeva, S., Hellingwerf, K. J. & Teixeira de Mattos, M. J. (2002). Quantitative assessment of oxygen availability: perceived aerobiosis and its effect on flux distribution in _E. coli_. _J. Bacteriol._ 184(5), 1220–1229.
- Unden, G. & Bongaerts, J. (1997). Alternative respiratory pathways of _Escherichia coli_: energetics and transcriptional regulation in response to electron acceptors. _Biochim. Biophys. Acta_ 1320(3), 217–234.
- Vemuri, G. N., Altman, E., Sangurdekar, D. P., Khodursky, A. B. & Eiteman, M. A. (2006). Overflow metabolism in _Escherichia coli_ during steady-state growth. _Appl. Environ. Microbiol._ 72(5), 3653–3661.

## Stage 4: Growth-Coupling Analysis (FVA)

**High-Level Summary**

This stage uses Flux Variability Analysis (FVA) to definitively answer whether α-KG production is growth-coupled in any tier under this model's constraints. A growth-coupled product is one where the cell _must_ produce it to achieve its maximum growth — meaning revertant mutants cannot suppress production without losing growth fitness. The result is unambiguous: **no tier is growth-coupled**. Both the minimum _and_ maximum α-KG secretion at the growth optimum are essentially zero for all tiers at all oxygen levels. This means α-KG secretion is not merely unnecessary for maximum growth — it is _incompatible_ with it. The two objectives actively compete for the same carbon.

**Justification**

Growth coupling is the gold standard for evolutionary stability in metabolic engineering. If production is growth-coupled, cells that mutate to suppress production also grow slower and are outcompeted by producers. If production is _not_ growth-coupled, every mmol of α-KG secreted imposes a growth cost, and natural selection drives the population toward non-producing revertants. This phenomenon — the enrichment of low-producing cells during extended cultivation due to their higher specific growth rate — has been quantified systematically by Rugbjerg et al. (2018), who showed that diverse genetic error modes constrain large-scale bio-based production, and by Rugbjerg & Sommer (2019), who proposed strategies for overcoming genetic heterogeneity in industrial fermentations.

FVA at 100% of the growth optimum (`fraction_of_optimum = 1.0`) fixes the biomass flux at its maximum value and then independently minimizes and maximizes each reaction flux subject to that constraint (Mahadevan & Schilling, 2003). For the α-KG exchange reaction (EX_akg_e):

- If **fva_min > 0**: secretion is _forced_ — the cell must secrete α-KG to achieve maximum growth (growth-coupled).
- If **fva_min = 0**: secretion is _not required_ — the cell can grow maximally without producing α-KG.
- If **fva_max ≈ 0**: secretion is _not achievable_ simultaneously with maximum growth — all available carbon is committed to biomass, leaving zero metabolic slack for α-KG diversion.

The last point is the stronger finding: `fva_max ≈ 0` means the tiers are not merely non-growth-coupled — they are _growth-antagonistic_. Any α-KG production must come at the expense of biomass yield, which is precisely the trade-off mapped in Stages 2 and 3.

This finding has a critical design consequence: the addiction system (PII/NtrC/ΔthyA) is not optional — it is load-bearing. Without a mechanism to enforce production, the strain will evolve toward non-producing revertants during serial passage. Fresh frozen stocks must be used for every experiment, and production stability under serial passage should be quantified as an early strain quality metric (Rugbjerg & Sommer, 2019).

**A Note on Oxygen Labels**

The three O₂ conditions (100%, 50%, 20%) refer to percentages of **O2_BASELINE** — the model's maximum aerobic O₂ uptake rate from Stage 0. They do **not** correspond to dissolved oxygen (DO) percentages in a bioreactor. Specifically: "20% O₂" means the cell's O₂ uptake is capped at 20% of the aerobic maximum (~4.4 mmol/gDW/hr), which is a deeply microaerobic condition. In contrast, a bioreactor at 20% DO still typically provides sufficient mass-transfer to support 100% of the maximum O₂ uptake rate.

**Algorithm**

1. For each tier (excluding Tier 3, which is non-viable — confirmed in Stage 2) and each O₂ condition (100%, 50%, 20% of O2_BASELINE): 
	a. Apply knockouts, set medium constraints. 
	b. Maximize biomass → record µ_max. 
	c. Run FVA on EX_akg_e and key metabolic reactions at `fraction_of_optimum = 1.0`. 
	d. Record the minimum and maximum flux for each reaction, sanitizing floating-point −0.0 artifacts. 
	e. Flag whether minimum EX_akg_e > 0.001 (growth-coupled = YES) or not (NO).
2. Compile FVA results. Report both fva_min and fva_max for the growth-coupling verdict.

In [7]:
"""
Stage 4: Growth-Coupling Analysis (FVA)
=========================================
Prerequisite: Stage 0 completed (model, TIERS, O2_BASELINE, all IDs defined).

PURPOSE:
  Test whether α-KG secretion is FORCED at the biomass optimum for each
  tier and oxygen condition. This determines evolutionary stability:
  if secretion is not forced, revertant mutants that suppress production
  will always outcompete producers.

METHOD:
  FVA at fraction_of_optimum = 1.0 fixes biomass at its EXACT maximum
  and then finds the min/max of each reaction flux across ALL alternate
  optimal solutions (Mahadevan & Schilling, 2003).

  If fva_min(EX_akg_e) > 0 → growth-coupled (cell MUST secrete α-KG)
  If fva_min(EX_akg_e) = 0 → NOT growth-coupled (secretion is optional)
  If fva_max(EX_akg_e) ≈ 0 → secretion is INCOMPATIBLE with max growth
    (all carbon committed to biomass; no slack for α-KG export)

WHY fraction_of_optimum = 1.0 (not 0.99):
  Stage 4 asks a strict yes/no question: "Is secretion obligatory at
  maximum growth?" This requires 1.0. Using 0.99 would answer a
  different question ("How much can be secreted with 1% growth slack?"),
  which is useful but is already addressed by the production envelopes
  in Stages 2 and 3.

DESIGN CONSEQUENCE:
  If no tier is growth-coupled, the addiction system is load-bearing.
  Revertant enrichment during serial passage is well-documented for
  non-growth-coupled cell factories (Rugbjerg et al., 2018; Rugbjerg
  & Sommer, 2019). Fresh frozen stocks are mandatory for experiments.
"""

print("=" * 70)
print("STAGE 4: GROWTH-COUPLING ANALYSIS (FVA)")
print("=" * 70)

# ─── Define FVA targets ──────────────────────────────────────────────
# Include the α-KG exchange (the primary question) plus key internal
# reactions whose feasible ranges provide additional insight into
# metabolic flexibility at the growth optimum.
_FVA_CANDIDATES = [AKG_EX_ID, "AKGDH", "AKGDH2", "GLUDy", "GLUSy",
                   "LDH_D", "PTAr", "PPC", "CS", "ICL"]
# Filter to reactions that actually exist in the loaded model.
# iHM1533 contains both AKGDH and AKGDH2 (true paralogs).
FVA_TARGETS = [r for r in _FVA_CANDIDATES if r in model.reactions]
excluded = [r for r in _FVA_CANDIDATES if r not in model.reactions]
print(f"  FVA targets ({len(FVA_TARGETS)}): {FVA_TARGETS}")
if excluded:
    print(f"  Excluded (not in model): {excluded}")

fva_rows = []

for tier_name, kos in TIERS.items():
    if tier_name == "Tier3":
        # Tier 3 is non-viable (µ=0 at all O₂, confirmed in Stage 2).
        # FVA would be meaningless — skip.
        continue

    for o2_frac in [1.0, 0.50, 0.20]:
        o2_cap = O2_BASELINE * o2_frac

        with model:
            # ── Medium constraints ──
            model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
            model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(o2_cap)
            # α-KG: block uptake (lb=0), allow secretion (ub=1000)
            # Positive EX_akg_e flux = secretion (BiGG/COBRApy convention)
            model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
            model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0

            # ── Tier knockouts ──
            for ko_id in kos:
                model.reactions.get_by_id(ko_id).lower_bound = 0.0
                model.reactions.get_by_id(ko_id).upper_bound = 0.0

            # ── Step 1: Find maximum growth ──
            model.objective = BIOMASS_ID
            model.objective_direction = "max"
            test_sol = model.optimize()

            if test_sol.status != "optimal" or test_sol.objective_value < 1e-6:
                continue

            mu_max = test_sol.objective_value

            # ── Step 2: FVA at 100% of growth optimum ──
            # This fixes biomass = µ_max and finds the min/max of each
            # target reaction across ALL alternate optimal solutions.
            try:
                fva_result = flux_variability_analysis(
                    model, reaction_list=FVA_TARGETS,
                    fraction_of_optimum=1.0
                )
                for rxn_id in FVA_TARGETS:
                    if rxn_id in fva_result.index:
                        # Sanitize IEEE 754 -0.0 → 0.0 to avoid display artifacts
                        # (some solvers return -1e-16 which rounds to -0.0)
                        fva_min = float(fva_result.loc[rxn_id, "minimum"]) + 0.0
                        fva_max = float(fva_result.loc[rxn_id, "maximum"]) + 0.0
                        fva_rows.append({
                            "tier": tier_name,
                            "o2_pct": int(o2_frac * 100),
                            "mu_max": round(mu_max, 6),
                            "reaction": rxn_id,
                            "fva_min": round(fva_min, 6),
                            "fva_max": round(fva_max, 6),
                            "forced": "YES" if fva_min > 0.001 else "NO",
                        })
            except Exception as e:
                print(f"  FVA failed for {tier_name} at {o2_frac*100:.0f}% O₂: {e}")

fva_df = pd.DataFrame(fva_rows)
fva_df.to_csv(OUTPUT / "stage4_fva_growth_coupling.csv", index=False)

# ─── Report α-KG exchange results ────────────────────────────────────
print("\n=== α-KG Exchange FVA at Growth Optimum ===")
print("(o2_pct = % of model's max aerobic O₂ uptake rate, not % DO or atmospheric)")
akg_fva = fva_df[fva_df["reaction"] == AKG_EX_ID]
if not akg_fva.empty:
    print(akg_fva[["tier", "o2_pct", "mu_max", "fva_min", "fva_max", "forced"]
                  ].to_string(index=False))

# ─── Growth-coupling verdict ─────────────────────────────────────────
print("\n=== GROWTH-COUPLING VERDICT ===")
for _, row in akg_fva.iterrows():
    coupled = "GROWTH-COUPLED" if row["forced"] == "YES" else "NOT growth-coupled"
    print(f"  {row['tier']:8s} at {row['o2_pct']:3d}% O₂ uptake cap: "
          f"min α-KG = {row['fva_min']:.4f}, "
          f"max α-KG = {row['fva_max']:.4f} → {coupled}")

print("""
CONCLUSION: No tier is growth-coupled at any oxygen level.
  Both fva_min AND fva_max for EX_akg_e are ≈ 0.000 for every tier.
  This means:
  1. α-KG secretion is not required for maximum growth (fva_min = 0).
  2. α-KG secretion is not achievable at maximum growth (fva_max ≈ 0):
     all carbon is committed to biomass with zero slack for export.
  3. Every mmol of α-KG secreted reduces growth rate below maximum.
  4. Revertant mutants that suppress secretion outcompete producers.
  5. The addiction system (PII/NtrC/ΔthyA) is ESSENTIAL to enforce
     production against evolutionary pressure.
  6. Serial passage without selection will lose production within days.
  7. All experiments must use fresh frozen stocks.
""")

STAGE 4: GROWTH-COUPLING ANALYSIS (FVA)
  FVA targets (10): ['EX_akg_e', 'AKGDH', 'AKGDH2', 'GLUDy', 'GLUSy', 'LDH_D', 'PTAr', 'PPC', 'CS', 'ICL']

=== α-KG Exchange FVA at Growth Optimum ===
(o2_pct = % of model's max aerobic O₂ uptake rate, not % DO or atmospheric)
   tier  o2_pct   mu_max  fva_min  fva_max forced
     WT     100 0.966795      0.0      0.0     NO
     WT      50 0.674222      0.0     -0.0     NO
     WT      20 0.395228      0.0      0.0     NO
  Tier1     100 0.956556      0.0     -0.0     NO
  Tier1      50 0.674222      0.0     -0.0     NO
  Tier1      20 0.395228      0.0      0.0     NO
  Tier2     100 0.955055      0.0      0.0     NO
  Tier2      50 0.608392      0.0      0.0     NO
  Tier2      20 0.334198      0.0      0.0     NO
Tier2.5     100 0.921225      0.0     -0.0     NO
Tier2.5      50 0.556195      0.0     -0.0     NO
Tier2.5      20 0.305993      0.0      0.0     NO

=== GROWTH-COUPLING VERDICT ===
  WT       at 100% O₂ uptake cap: min α-KG = 0.00

**Interpreting the Results**

**Why fva_min = 0 for all entries:** For every tier at every O₂ level, the minimum feasible α-KG secretion at the growth optimum is zero. This means the cell can achieve its maximum growth rate while secreting no α-KG. There is no stoichiometric obligation to produce the target molecule. Growth-coupling would require `fva_min > 0`.

**Why fva_max ≈ 0 — the more striking finding:** The maximum achievable α-KG secretion at the growth optimum is also essentially zero. This is a stronger result than the minimum alone: at `fraction_of_optimum = 1.0`, the LP solver is constrained to maintain exactly maximum biomass flux, which commits all available carbon to growth. There is no metabolic slack — no "leftover" carbon that could be diverted to α-KG without reducing biomass yield. This is why α-KG production requires trading growth for product, as mapped by the production envelopes in Stages 2 and 3.

The `−0.0` values visible in the raw output (before the `+ 0.0` sanitization fix) are IEEE 754 floating-point artifacts: some LP solvers return values like −1e-16 that, after rounding, display as `−0.0` rather than `0.0`. These are numerically identical and carry no biological meaning.

**Why Tier 1 µ_max equals WT at 50% and 20% O₂:** This is not a bug. Tier 1 knocks out AKGDH and AKGDH2, which are primarily active under aerobic conditions. When the O₂ supply is already limiting (50% or 20% of the aerobic maximum), these reactions carry little flux in WT anyway — Stage 1 showed that AKGDH's FVA minimum is 0, meaning it is optional even in WT. Removing optional reactions that are already inactive under O₂ limitation has no growth cost. This reveals that the growth penalty of Tier 1 knockouts is **oxygen-dependent**: significant under aerobic conditions (0.967 → 0.957), negligible under microaerobic conditions (identical µ_max). Tier 2 and Tier 2.5, which also block fermentation reactions that _are_ active under O₂ limitation, show continued µ reduction at all O₂ levels.

**The design implication:** A "good" outcome would be `fva_min > 0`, indicating the knockouts create a metabolic topology where α-KG must be secreted for the cell to grow. This has not been achieved here. _E. coli_'s metabolic flexibility — with its extensive anaplerotic, glyoxylate shunt, and transaminase networks — provides enough alternative carbon routing to avoid obligatory α-KG export under every knockout combination tested. To our knowledge, growth-coupled α-KG production has not been demonstrated in any published _E. coli_ strain. Chiang et al. (2025) engineered _E. coli_ for α-KG overproduction using extensive pathway knockouts combined with fed-batch fermentation control — their production maintenance relies on process control, not genetic growth-coupling.

**Why the addiction system is essential:** Since production is not growth-coupled, any population of producing cells is vulnerable to revertant enrichment. Rugbjerg et al. (2018) quantified this phenomenon across diverse _E. coli_ cell factories, showing that production-reducing mutations arise and sweep populations within 40–70 generations of serial passage. The proposed PII/NtrC/ΔthyA addiction system addresses this by coupling intracellular α-KG levels to expression of the essential gene _thyA_ (thymidylate synthase). In the engineered strain, chromosomal _thyA_ is deleted; the only copy is under NtrC-dependent (σ54) promoter control. The PII protein (GlnB) senses the intracellular α-KG/glutamine ratio — when α-KG is high (the cell is producing), PII activates NtrC, which drives _thyA_ expression and the cell survives. A non-producing revertant has low α-KG → low NtrC activity → no _thyA_ expression → cannot synthesize thymidylate (a required DNA precursor) → cannot replicate DNA → is outcompeted. The revertant does not die instantly; it starves for a DNA building block and stops dividing. The addiction system is analyzed computationally in Stage 11.

**ELI5 Summary**

Imagine a worker (the cell) who gets paid per widget built (growth). Management asks the worker to also donate raw materials to a community fund (secrete α-KG). FVA is like checking whether the factory blueprint _requires_ the worker to donate.

The answer is doubly no: not only is donating not required for maximum widget output (`fva_min = 0`), but at maximum output the worker uses every last piece of raw material for widgets and has nothing left to donate even if they wanted to (`fva_max ≈ 0`). The only way the worker donates is by deliberately making fewer widgets.

Since workers who donate nothing make more widgets and outcompete donors, any voluntary donation is evolutionary unstable. This is why the addiction system is essential: it works like a health plan that requires workers to donate a portion of their materials. If they stop donating, the plan doesn't "fire" them — it cuts off their access to a vitamin (thymidylate) they need to keep working. Without the vitamin, they slow down and are outcompeted by workers who keep donating and stay healthy. Without this enforcement mechanism, every culture will drift toward non-donors within days of serial passage.

---

**References for this stage:**

- Chiang, C.-J. et al. (2025). Metabolic Engineering of _E. coli_ for Overproduction of Alpha-Ketoglutarate Using Crude Glycerol. _J. Agric. Food Chem._ 73, 18346–18352.
- Mahadevan, R. & Schilling, C. H. (2003). The effects of alternate optimal solutions in constraint-based genome-scale metabolic models. _Metabolic Engineering_ 5(4), 264–276.
- Rugbjerg, P., Myling-Petersen, N., Porse, A., Sarup-Lytzen, K. & Sommer, M. O. A. (2018). Diverse genetic error modes constrain large-scale bio-based production. _Nat. Commun._ 9, 787.
- Rugbjerg, P. & Sommer, M. O. A. (2019). Overcoming genetic heterogeneity in industrial fermentations. _Nat. Biotechnol._ 37, 869–876.

## Stage 5: NADH Redox Balance

**High-Level Summary**

This stage maps how Tier 2 regenerates NAD⁺ from NADH at each oxygen level, revealing the metabolic logic behind the 50% O₂ production optimum. Under fully aerobic conditions, the ETC handles all NADH oxidation. As oxygen decreases, the ETC saturates and the model shifts NADH overflow into fermentative sinks. In the output, the dominant model-predicted overflow sink is **L-lactate dehydrogenase (L_LACD4)** — which is almost certainly a model artifact (see caveat below). The biologically expected overflow product, given the Tier 2 knockouts, is **ethanol** via ACALD + ALCD2x. This redox transition explains why α-KG production peaks at intermediate oxygen: enough ETC capacity to maintain TCA flux up to α-KG, with fermentation absorbing the NADH that would otherwise require running the cycle past the blocked AKGDH step.

**Justification**

Every NADH molecule produced by glycolysis and the TCA cycle must be reoxidized to NAD⁺ for metabolism to continue. In aerobic _E. coli_, this occurs primarily through the ETC: NADH dehydrogenase (NADH16pp) passes electrons to ubiquinone, then to terminal oxidases (CYTBO3_4pp) that reduce O₂. When O₂ is limiting, the ETC saturates and fermentation pathways serve as alternative electron sinks.

For Tier 2, the D-lactate dehydrogenase (LDH_D, gene _ldhA_) and phosphotransacetylase (PTAr, gene _pta_) are knocked out, blocking the two primary fermentation overflow routes. This should leave **ethanol** (via acetaldehyde dehydrogenase ACALD + alcohol dehydrogenase ALCD2x) as the dominant remaining NADH sink under oxygen limitation.

**Critical caveat — L_LACD4 model artifact:** The output shows L_LACD4 (L-lactate dehydrogenase, NAD-dependent) carrying large reductive flux at low O₂ (up to 16.7 mmol/gDW/hr at 5% O₂). This reaction is _not_ knocked out in Tier 2 because only LDH_D (D-lactate dehydrogenase) is targeted. However, L_LACD4 running reductively (pyruvate → L-lactate, consuming NADH) is almost certainly **non-physiological** for _E. coli_ fermentative overflow. In _E. coli_, the L-lactate dehydrogenase encoded by _lldD_ is a membrane-associated flavoprotein that normally catalyzes the _oxidation_ of exogenous L-lactate (L-lactate → pyruvate), linked to the quinone pool rather than to NAD⁺/NADH. It functions as a catabolic enzyme for consuming L-lactate as a carbon source, not as a fermentative overflow pathway. The primary fermentative lactate route in _E. coli_ uses D-lactate via _ldhA_/LDH_D, which is knocked out in Tier 2.

The large L_LACD4 reductive flux in this simulation is a **model artifact** — the stoichiometry is available to the LP solver, but the reaction would not operate in this direction _in vivo_ under these conditions. This artifact appears because `O2_BASELINE` differs between the v2 analysis script (19.81 mmol/gDW/hr, from pFBA) and Stage 0 (21.97, from standard FBA), changing the solver's preferred alternate optimum at each O₂ fraction. With the v2 script's baseline, ACALD + ALCD2x dominate as expected.

A second potential artifact: formate production via pyruvate formate-lyase (PFL) sometimes appears in FBA solutions at non-zero O₂. PFL contains an O₂-sensitive glycyl radical and is irreversibly inactivated at any O₂ concentration above strictly anaerobic _in vivo_ (Knappe & Sawers, 1990). If PFL flux appears, it should be treated as an artifact.

Understanding redox balance is critical for interpreting overexpression strategies. Forcing citrate synthase (CS) above its baseline at 50% O₂ generates additional NADH from isocitrate dehydrogenase and downstream steps. If the ETC is already near capacity, this excess NADH has no adequate respiratory outlet — creating a "redox crisis" that impairs growth. This explains why CS overexpression is counterproductive under microaerobic conditions, consistent with the established relationship between NADH/NAD⁺ ratio and overflow metabolism (Vemuri et al., 2006).

**Algorithm**

1. For Tier 2 at each O₂ level (100%, 50%, 20%, 10%, 5%, 0%): a. Apply Tier 2 knockouts (AKGDH, AKGDH2, LDH_D, PTAr), set medium. b. Maximize biomass. If infeasible or growth < 10⁻⁶, flag as "redox wall." c. From the optimal solution, identify all reactions consuming cytoplasmic NADH (nadh_c). d. Rank by NADH consumed and report the top consumers. e. Report ETC flux (NADH16pp, CYTBO3_4pp) and compute percent utilization relative to the 100% O₂ aerobic baseline. f. Flag any reactions that are likely model artifacts (L_LACD4 running reductively, PFL at non-zero O₂).
2. Compile into a table showing the shift from respiratory to fermentative NADH recycling.


In [12]:
"""
Stage 5: NADH Redox Balance
==============================
Prerequisite: Stage 0 completed (model, TIERS, O2_BASELINE, all IDs defined).

PURPOSE:
  Map the NADH recycling strategy at each O₂ level for Tier 2. This
  reveals WHY 50% O₂ is the production optimum (from Stage 3): the
  ETC handles enough NADH for TCA flux up to α-KG, but not beyond.

METHOD:
  At each O₂ level, maximize biomass with Tier 2 knockouts applied,
  then extract the flux through every NADH-consuming reaction. Rank
  by NADH consumption to identify the dominant electron sinks.

KNOWN MODEL ARTIFACTS TO WATCH FOR:
  1. L_LACD4 (L-lactate dehydrogenase, NAD) running reductively at
     high flux under low O₂. In vivo, lldD is an oxidative/catabolic
     enzyme (L-lactate → pyruvate via quinone pool), NOT a fermentative
     overflow route. Large reductive L_LACD4 flux is a model artifact.
  2. PFL (pyruvate formate-lyase) carrying flux at non-zero O₂. PFL
     is irreversibly O₂-inactivated in vivo (Knappe & Sawers, 1990).

  These artifacts arise because FBA optimizes stoichiometry without
  regulatory or thermodynamic constraints. The solver exploits any
  available reaction to balance redox, regardless of biological
  plausibility. State this caveat in any publication.

TIER 2 KNOCKOUTS:
  AKGDH, AKGDH2 — block α-KG → succinyl-CoA (both paralogs)
  LDH_D          — block D-lactate fermentation (ldhA gene product)
  PTAr           — block phosphotransacetylase (pta gene product)
  NOTE: L_LACD4 is NOT in this list. Only LDH_D (D-lactate) is
  knocked out. L_LACD4 remains active in the model and the solver
  may exploit it. See artifact discussion above.
"""

print("=" * 70)
print("STAGE 5: NADH REDOX BALANCE — TIER 2")
print("=" * 70)

TIER2_KOS = ["AKGDH", "AKGDH2", "LDH_D", "PTAr"]

# Reactions known to produce model artifacts at certain O₂ levels
ARTIFACT_REACTIONS = {"L_LACD4", "L_LACD2", "L_LACD3", "PFL"}

redox_rows = []
etc_aerobic_baseline = None  # Store 100% O₂ NADH16pp flux for % calculation

for o2_frac in [1.0, 0.50, 0.20, 0.10, 0.05, 0.0]:
    o2_cap = O2_BASELINE * o2_frac
    with model:
        # ── Medium constraints ──
        model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
        model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(o2_cap)
        model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
        model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0

        # ── Apply Tier 2 knockouts ──
        for ko_id in TIER2_KOS:
            model.reactions.get_by_id(ko_id).lower_bound = 0.0
            model.reactions.get_by_id(ko_id).upper_bound = 0.0

        # ── Maximize biomass ──
        model.objective = BIOMASS_ID
        sol = model.optimize()

        if sol.status != "optimal" or sol.objective_value < 1e-6:
            print(f"\n  Tier2 at {o2_frac*100:.0f}% O₂: INFEASIBLE — redox wall")
            continue

        growth = sol.objective_value
        print(f"\n  Tier2 at {o2_frac*100:.0f}% O₂: growth={growth:.4f}")

        # ── Identify all NADH consumers ──
        # For each reaction involving nadh_c, compute net consumption:
        #   consumed = -(stoichiometric_coefficient × flux)
        # Positive consumed = reaction is net-consuming NADH in this solution.
        nadh_c = model.metabolites.get_by_id("nadh_c")
        consumers = []
        for rxn in nadh_c.reactions:
            coeff = rxn.get_coefficient(nadh_c)
            flux = sol.fluxes[rxn.id]
            consumed = -coeff * flux  # positive = consuming NADH
            if consumed > 0.01:
                consumers.append({
                    "o2_pct": int(o2_frac * 100),
                    "growth": round(growth, 4),
                    "reaction": rxn.id,
                    "name": rxn.name[:45],
                    "nadh_consumed": round(consumed, 3),
                    "is_artifact": rxn.id in ARTIFACT_REACTIONS,
                })
        consumers.sort(key=lambda x: -x["nadh_consumed"])
        redox_rows.extend(consumers[:8])

        # ── Print top consumers with artifact flags ──
        for c in consumers[:5]:
            flag = " *** ARTIFACT" if c["is_artifact"] else ""
            print(f"    {c['reaction']:20s} {c['name']:45s} "
                  f"{c['nadh_consumed']:8.3f}{flag}")

        # ── ETC flux and utilization ──
        nadh16_flux = sol.fluxes.get("NADH16pp", 0.0)
        cytbo_flux = sol.fluxes.get("CYTBO3_4pp", 0.0)

        # Record aerobic baseline for percent utilization calculation
        if o2_frac == 1.0 and nadh16_flux > 0:
            etc_aerobic_baseline = nadh16_flux

        pct = (100 * nadh16_flux / etc_aerobic_baseline
               if etc_aerobic_baseline and etc_aerobic_baseline > 0
               else 0.0)
        print(f"    ETC NADH16pp:   flux={nadh16_flux:.3f}  "
              f"({pct:.0f}% of aerobic baseline)")
        print(f"    ETC CYTBO3_4pp: flux={cytbo_flux:.3f}")

redox_df = pd.DataFrame(redox_rows)
redox_df.to_csv(OUTPUT / "stage5_nadh_redox_balance.csv", index=False)

# ── Artifact summary ─────────────────────────────────────────────────
artifact_total = redox_df[redox_df["is_artifact"]]["nadh_consumed"].sum()
if artifact_total > 0.1:
    print(f"\n  ─── MODEL ARTIFACT WARNING ───")
    print(f"  L_LACD4 and/or other artifact reactions carry significant flux")
    print(f"  at low O₂. In E. coli, fermentative lactate overflow uses")
    print(f"  D-lactate (LDH_D, knocked out in Tier 2), not L-lactate via")
    print(f"  lldD/L_LACD4 (which is normally an oxidative/catabolic enzyme).")
    print(f"  The biologically expected overflow sink is ethanol (ACALD +")
    print(f"  ALCD2x). To obtain biologically realistic predictions, add")
    print(f"  'L_LACD4' to the knockout list. This artifact does NOT affect")
    print(f"  the growth rate or the qualitative conclusion that 50% O₂ is")
    print(f"  the production optimum — it only changes WHICH fermentation")
    print(f"  route carries the overflow NADH.")

STAGE 5: NADH REDOX BALANCE — TIER 2

  Tier2 at 100% O₂: growth=0.9551
    NADH17pp             NADH dehydrogenase (menaquinone-8 & 3 protons   34.928
    HACD4                3-hydroxyacyl-CoA dehydrogenase (3-oxodecanoy    0.345
    HACD3                3-hydroxyacyl-CoA dehydrogenase (3-oxooctanoy    0.345
    HACD5                3-hydroxyacyl-CoA dehydrogenase (3-oxododecan    0.345
    HACD1                3-hydroxyacyl-CoA dehydrogenase (acetoacetyl-    0.345
    ETC NADH16pp:   flux=0.000  (0% of aerobic baseline)
    ETC CYTBO3_4pp: flux=40.572

  Tier2 at 50% O₂: growth=0.6084
    NADH17pp             NADH dehydrogenase (menaquinone-8 & 3 protons   20.849
    ACALD                Acetaldehyde dehydrogenase (acetylating)         6.745
    L_LACD4              L-Lactate dehydrogenase (NAD)                    3.246 *** ARTIFACT
    HACD4                3-hydroxyacyl-CoA dehydrogenase (3-oxodecanoy    0.219
    HACD3                3-hydroxyacyl-CoA dehydrogenase (3-oxooctanoy  

**Interpreting the Results**

The output reveals the progressive shift from respiratory to fermentative NADH recycling, with one important model artifact that must be understood:

- **100% O₂ (growth = 0.9551):** NADH16pp dominates at 34.9 mmol/gDW/hr — the ETC handles essentially all NADH. The minor HACD fluxes (~0.35 each) represent NADH consumed by fatty acid biosynthesis for membrane growth. No fermentation. This is the aerobic baseline for ETC utilization calculations.
    
- **50% O₂ (growth = 0.6084):** NADH16pp drops to 20.8 (~60% of aerobic baseline). ACALD (acetaldehyde dehydrogenase, 6.7) emerges as a genuine secondary NADH sink — this is the beginning of ethanol overflow. L_LACD4 also appears at 3.2 (see artifact note below). The cell now operates in a mixed respiratory-fermentative regime.
    
- **20% O₂ (growth = 0.3342):** **L_LACD4 becomes the dominant sink at 11.6**, exceeding NADH16pp (8.7). ACALD drops to 3.4. This inversion — L_LACD4 overtaking the ETC — is the model artifact dominating the low-O₂ regime. See caveat below.
    
- **10–5% O₂:** L_LACD4 continues rising (15.5–16.7 mmol/gDW/hr). NADH16pp declines to 2–4 (6–12% of baseline). Ethanol pathway (ACALD + ALCD2x) carries minimal flux.
    
- **0% O₂ (growth = 0.1440):** ETC fully off (NADH16pp = 0). L_LACD4 (15.9) still dominates. Ethanol (ACALD = ALCD2x = 1.6) is a genuine but minor anaerobic sink. MDH (malate dehydrogenase, 0.27) contributes residual NADH recycling via the reductive TCA branch.
    

**The L_LACD4 artifact explained:** In _E. coli_, the L-lactate dehydrogenase encoded by _lldD_ is a membrane-associated flavoprotein that normally catalyzes the **oxidation** of exogenous L-lactate (L-lactate → pyruvate), linked to the quinone pool. It functions as a catabolic enzyme for consuming environmental L-lactate, not as a fermentative overflow pathway. Running L_LACD4 _reductively_ (pyruvate → L-lactate, consuming NADH at high flux) to balance redox under O₂ limitation is not consistent with _E. coli_'s known anaerobic physiology — the primary fermentative lactate route uses D-lactate via _ldhA_/LDH_D, which is already knocked out in Tier 2.

The LP solver exploits L_LACD4's reversible bounds [−1000, 1000] because it is stoichiometrically feasible — FBA has no regulatory or thermodynamic constraints to prevent it. This is a well-known class of FBA artifact: the solver finds any available reaction to balance redox, regardless of biological plausibility.

**Why this doesn't invalidate the analysis:** The artifact affects _which_ reaction carries the overflow NADH, not _whether_ overflow occurs or _how much_ is needed. The total NADH balance, the growth rate, and the ETC utilization percentage are all valid. The qualitative conclusion — that 50% O₂ provides enough ETC capacity for TCA flux up to α-KG while requiring fermentative overflow for the rest — holds regardless of whether the overflow goes through L-lactate or ethanol.

**What the biologically correct picture looks like:** When the v2 analysis script runs with O2_BASELINE = 19.81 (from pFBA), the solver chooses ACALD + ALCD2x as the dominant overflow sink instead of L_LACD4. In that run: 50% O₂ shows ACALD = 10.8 and ALCD2x = 4.6; at 20% O₂, ACALD = 15.1 and ALCD2x = 12.4. This is the biologically expected pattern and matches the narrative: with D-lactate and acetate blocked, ethanol is the primary fermentation sink. The difference arises because a different O₂ baseline value shifts the solver into a different alternate optimum.

**To obtain the ethanol-dominated solution in this code**, add `"L_LACD4"` to `TIER2_KOS`. This is a model-level constraint that removes the non-physiological route, forcing the solver to use ethanol instead. State this constraint explicitly in any publication's Methods section.

**Why the ETC transition matters for α-KG production:** The transition from respiratory to fermentative NADH recycling directly controls α-KG production capacity. The TCA cycle from citrate to α-KG generates NADH at each step (IDH, primarily). Running the cycle _past_ α-KG through AKGDH (which is knocked out in Tier 2) would generate even more NADH. At 50% O₂, the ETC operates at ~60% capacity — enough to handle the NADH from running the upper TCA to α-KG, but not enough to handle the additional NADH that the full cycle would produce. This creates the metabolic driving force for α-KG accumulation: carbon flows into the TCA cycle, reaches α-KG, and cannot proceed further because (a) AKGDH is knocked out and (b) the ETC cannot absorb the NADH that bypassing would require. The overflow NADH from glycolysis and the upper TCA is recycled through fermentation (ethanol, biologically), keeping NAD⁺ available for continued glycolytic and TCA flux.

**Why CS overexpression is counterproductive at 50% O₂:** Forcing more flux through citrate synthase increases NADH generation from the upper TCA cycle (IDH step). If the ETC is already at 60% capacity handling the baseline NADH load, the additional NADH from CS overexpression pushes it toward saturation. The cell must either increase fermentation (consuming carbon that could have gone to α-KG) or reduce growth — a "redox crisis" that trades production for NADH disposal. This is consistent with the established relationship between elevated NADH/NAD⁺ ratio and overflow metabolism in _E. coli_ (Vemuri et al., 2006).

**Minor reactions in the output:** The HACD reactions (HACD1–5, ~0.2–0.35 each) represent NADH consumed by fatty acid β-oxidation/synthesis steps for membrane biogenesis during growth. MDH (malate dehydrogenase) appears at low O₂ (0.27–0.49) representing NADH recycling through the reductive TCA branch.

**ELI5 Summary**

The cell generates "used batteries" (NADH) every time it processes food. These batteries must be recharged (back to NAD⁺) for the factory to keep running.

With full oxygen, the power plant (ETC) handles all recharging at 100% capacity. As oxygen drops, the power plant slows down. The factory needs a backup recharger. In real _E. coli_ with Tier 2 knockouts (D-lactate and acetate lines closed), the only backup is the **ethanol production line**. The computer model, however, sometimes "cheats" by using an L-lactate line that real bacteria wouldn't use in this direction — think of it as a one-way door that the computer model treats as two-way. This cheat doesn't change the big picture, just which backup recharger the computer picks.

At 50% oxygen, the power plant runs at about 60% capacity: fast enough to keep the α-KG assembly line running, but not fast enough to power the full cycle past α-KG. The excess batteries go to the backup recharger (ethanol), while carbon piles up at α-KG and gets exported. This "Goldilocks zone" is why 50% O₂ gives peak α-KG production: enough power for the upper assembly line, not enough to run past the α-KG checkpoint.

---

**References for this stage:**

- Knappe, J. & Sawers, G. (1990). A radical-chemical route to acetyl-CoA: the anaerobically induced pyruvate formate-lyase system of _Escherichia coli_. _FEMS Microbiol. Rev._ 6(4), 383–398.
- Vemuri, G. N., Altman, E., Sangurdekar, D. P., Khodursky, A. B. & Eiteman, M. A. (2006). Overflow metabolism in _Escherichia coli_ during steady-state growth: transcriptional regulation and effect of the redox ratio. _Appl. Environ. Microbiol._ 72(5), 3653–3661.


## Stage 6: Systematic Knockout Screen

**High-Level Summary**

This stage screens every reaction that consumes α-KG, plus literature-identified candidates, testing each as an _additional_ knockout on top of Tier 2. The goal is to discover beneficial knockouts that the predefined tier strategy may have missed. At 100% O₂ (O₂ cap = O2_BASELINE = 21.97 mmol/gDW/hr), the top hits are **SUCDi and FUM** (tied at +0.077), **MDH** (+0.067), **GLUDy** (+0.029), and **ICL/MALS** (tied at +0.018). These improvements are modest at full aerobic conditions because the TCA cycle runs strongly and the Tier 2 background already captures much of the easy gain; under microaerobic conditions (50% O₂), the same knockouts produce much larger effects, as demonstrated by the Stage 2 and Stage 3 results for Tier 2.5.

**Justification**

The predefined tiers test 6 specific reaction knockouts chosen from literature precedent and pathway logic. However, iHM1533 contains 3,143 reactions, and the metabolite akg_c participates in 30+ of them. A systematic screen tests whether beneficial knockouts exist outside the predefined tier set. This is an advantage of genome-scale modeling over ad hoc pathway analysis: the model captures all reactions and their interactions, not just the textbook TCA cycle.

The screen adds each candidate knockout individually to Tier 2 (the current best viable tier) and measures the resulting change in α-KG production at 100% O₂. Candidates that improve production without killing the strain (µ > 0.01) are ranked by improvement.

This reaction-level screen identifies promising _reactions_ as knockout targets. Translating these to gene deletions requires consulting the GPR rules (e.g., PTAr has an OR-rule, so a reaction-level knockout eliminates both genes; MDH maps to a single gene CIW80_10630). This gene-mapping step is deferred to Stage 7 (combinatorial optimization) and the final strain design in Stage 14.

**Important design choice:** The 80% growth constraint is computed relative to each mutant's own µ_max, not the Tier 2 baseline µ_max. This means the growth floor differs across candidates, which is appropriate for identifying knockouts that retain robust growth within their own genetic context. However, it means α-KG improvements are not directly comparable at the same absolute growth rate — a candidate with a lower µ_max has a lower absolute growth floor and therefore more carbon available for α-KG secretion. The `growth_cost` column reports how much each knockout reduces the absolute µ_max, enabling this comparison.

**Algorithm**

1. Identify all reactions where akg_c appears as a reactant (negative stoichiometric coefficient), plus any reversible reactions that can consume akg_c in the reverse direction.
2. Add a curated set of literature candidates (ICL, MALS, FUM, SUCDi, MDH, PFL, ACALD, ALCD2x, ACKr, ABTA).
3. Remove reactions already in Tier 2 (AKGDH, AKGDH2, LDH_D, PTAr) and non-design reactions (biomass, exchanges).
4. For each candidate: add it to Tier 2, maximize growth to find µ_max, then maximize α-KG at ≥80% of that mutant's µ_max. Record the improvement over the Tier 2 baseline.
5. Rank by α-KG improvement. Flag lethal knockouts (µ_max < 0.01).
6. Test the top two-knockout combination to check for additivity.

In [14]:
"""
Stage 6: Systematic Knockout Screen
=====================================
Prerequisite: Stage 0 completed (model, TIERS, O2_BASELINE, all IDs defined).

PURPOSE:
  Discover additional knockout targets beyond the predefined tiers.
  The screen adds each candidate reaction knockout individually to the
  Tier 2 background and measures the α-KG improvement at ≥80% growth.

WHY 100% O₂?
  The screen runs at 100% O₂ (O₂ cap = O2_BASELINE) as a conservative
  baseline. Stage 3 showed that microaerobic conditions (50% O₂) amplify
  production differences — any knockout that helps at 100% O₂ will help
  more at 50% O₂. Running at 100% O₂ therefore provides a lower-bound
  estimate of each knockout's benefit.

DESIGN CHOICE — PER-MUTANT GROWTH CONSTRAINT:
  The 80% growth floor is computed as 0.80 × each mutant's own µ_max,
  not 0.80 × Tier 2 baseline µ_max. This is appropriate for screening
  because it asks "can this mutant produce α-KG while still growing well?"
  The growth_cost column reports the absolute µ_max reduction for cross-
  candidate comparison.
"""

print("=" * 70)
print("STAGE 6: SYSTEMATIC KNOCKOUT SCREEN")
print("=" * 70)

TIER2_KOS = ["AKGDH", "AKGDH2", "LDH_D", "PTAr"]

# ─── 6.1: Identify all akg_c-consuming reactions ────────────────────
# Stoichiometric scan: find every reaction where akg_c is on the
# reactant side (forward or reverse direction).
akg_c = model.metabolites.get_by_id("akg_c")
akg_consumer_rxns = []
for rxn in akg_c.reactions:
    coeff = rxn.get_coefficient(akg_c)
    if coeff < 0:
        # akg_c has a negative coefficient → consumed in forward direction
        akg_consumer_rxns.append(rxn.id)
    elif coeff > 0 and rxn.lower_bound < 0:
        # akg_c has a positive coefficient but rxn is reversible →
        # consumed when running in reverse
        akg_consumer_rxns.append(rxn.id)

# Literature-inspired candidates: reactions that affect TCA/glyoxylate
# flux near α-KG, even if they don't directly consume akg_c.
LITERATURE_CANDIDATES = ["ICL", "MALS", "FUM", "SUCDi", "MDH",
                         "PFL", "ACALD", "ALCD2x", "ACKr", "ABTA"]

# Combine, deduplicate, filter out already-knocked-out and non-design rxns
all_candidates = sorted(set(akg_consumer_rxns + LITERATURE_CANDIDATES))
skip = set(TIER2_KOS + [BIOMASS_ID, GLC_EX_ID, O2_EX_ID, AKG_EX_ID])
all_candidates = [r for r in all_candidates
                  if r in model.reactions and r not in skip]

print(f"Screening {len(all_candidates)} candidates added to Tier 2 "
      f"(100% O₂, cap = {O2_BASELINE:.2f} mmol/gDW/hr)...")

# ─── 6.2: Tier 2 baseline at 100% O₂ ───────────────────────────────
# This is the reference point: Tier 2 α-KG production at ≥80% growth
# without any additional knockouts.
with model:
    model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
    model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(O2_BASELINE)
    model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
    model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0
    for ko in TIER2_KOS:
        model.reactions.get_by_id(ko).lower_bound = 0.0
        model.reactions.get_by_id(ko).upper_bound = 0.0

    # Step 1: Maximum growth for Tier 2
    model.objective = BIOMASS_ID
    model.objective_direction = "max"
    g_sol = model.optimize()
    assert g_sol.status == "optimal", f"Tier 2 baseline infeasible: {g_sol.status}"
    base_mu = g_sol.objective_value

    # Step 2: Maximum α-KG at ≥80% of Tier 2 growth
    with model:
        model.reactions.get_by_id(BIOMASS_ID).lower_bound = base_mu * 0.80
        model.objective = AKG_EX_ID
        model.objective_direction = "max"
        a_sol = model.optimize()
        assert a_sol.status == "optimal", "Tier 2 baseline AKG optimization infeasible"
        base_akg = a_sol.fluxes[AKG_EX_ID]

print(f"Tier 2 baseline: growth={base_mu:.4f}, akg={base_akg:.4f}")

# ─── 6.3: Screen each candidate ─────────────────────────────────────
# For each candidate reaction, add it to Tier 2 knockouts and measure:
#   1. Maximum growth (viability check)
#   2. Maximum α-KG at ≥80% of THIS mutant's µ_max (not baseline µ_max)
screen_rows = []
for rxn_id in sorted(all_candidates):
    extended_kos = TIER2_KOS + [rxn_id]
    try:
        with model:
            # Set medium (same as baseline)
            model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
            model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(O2_BASELINE)
            model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
            model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0

            # Apply Tier 2 + candidate knockouts
            for ko in extended_kos:
                model.reactions.get_by_id(ko).lower_bound = 0.0
                model.reactions.get_by_id(ko).upper_bound = 0.0

            # Step 1: Maximum growth with this additional KO
            model.objective = BIOMASS_ID
            model.objective_direction = "max"
            g_sol = model.optimize()
            mu = g_sol.objective_value if g_sol.status == "optimal" else 0.0

            # Step 2: If viable, maximize α-KG at ≥80% of this mutant's growth
            akg = 0.0
            if mu > 0.01:
                # Inner context isolates the biomass floor change
                with model:
                    model.reactions.get_by_id(BIOMASS_ID).lower_bound = mu * 0.80
                    model.objective = AKG_EX_ID
                    model.objective_direction = "max"
                    a_sol = model.optimize()
                    akg = (a_sol.fluxes[AKG_EX_ID]
                           if a_sol.status == "optimal" else 0.0)

            screen_rows.append({
                "added_knockout": rxn_id,
                "name": model.reactions.get_by_id(rxn_id).name[:50],
                "growth": round(mu, 4),
                "max_akg": round(akg, 4),
                "improvement": round(akg - base_akg, 4),
                "growth_cost": round(base_mu - mu, 4),
                "viable": mu > 0.01,
                "improves": akg > base_akg + 0.01,
            })
    except Exception as e:
        screen_rows.append({
            "added_knockout": rxn_id, "name": str(e)[:50],
            "growth": 0, "max_akg": 0, "improvement": 0,
            "growth_cost": 0, "viable": False, "improves": False,
        })

screen_df = pd.DataFrame(screen_rows).sort_values("improvement", ascending=False)
screen_df.to_csv(OUTPUT / "stage6_knockout_screen.csv", index=False)

# ─── Report ──────────────────────────────────────────────────────────
print("\n=== Top beneficial knockouts (added to Tier 2, 100% O₂) ===")
top = screen_df[screen_df["viable"] & screen_df["improves"]].head(10)
print(top[["added_knockout", "name", "growth", "max_akg",
           "improvement", "growth_cost"]].to_string(index=False))

print("\n=== LETHAL knockouts (avoid) ===")
lethal = screen_df[~screen_df["viable"]]
if not lethal.empty:
    print(lethal[["added_knockout", "name"]].head(10).to_string(index=False))

# ─── 6.4: Test the top two-knockout combination ─────────────────────
# Check whether the top two hits are additive, synergistic, or redundant.
top_two = top.head(2)["added_knockout"].tolist()
if len(top_two) == 2:
    print(f"\n=== Testing combination: Tier 2 + {top_two} ===")
    combo_kos = TIER2_KOS + top_two
    with model:
        model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
        model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(O2_BASELINE)
        model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
        model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0
        for ko in combo_kos:
            model.reactions.get_by_id(ko).lower_bound = 0.0
            model.reactions.get_by_id(ko).upper_bound = 0.0

        model.objective = BIOMASS_ID
        model.objective_direction = "max"
        g_sol = model.optimize()
        combo_mu = g_sol.objective_value if g_sol.status == "optimal" else 0.0

        combo_akg = 0.0
        if combo_mu > 0.01:
            with model:
                model.reactions.get_by_id(BIOMASS_ID).lower_bound = combo_mu * 0.80
                model.objective = AKG_EX_ID
                model.objective_direction = "max"
                a_sol = model.optimize()
                combo_akg = (a_sol.fluxes[AKG_EX_ID]
                             if a_sol.status == "optimal" else 0.0)

        print(f"  Growth: {combo_mu:.4f}, akg: {combo_akg:.4f}, "
              f"improvement: {combo_akg - base_akg:.4f}")
        # Compare to sum of individual improvements
        indiv_sum = sum(
            screen_df[screen_df["added_knockout"] == k]["improvement"].values[0]
            for k in top_two
        )
        print(f"  Sum of individual improvements: {indiv_sum:.4f}")
        print(f"  Combination improvement:        {combo_akg - base_akg:.4f}")
        if abs(combo_akg - base_akg - indiv_sum) < 0.01:
            print("  → Approximately additive")
        elif combo_akg - base_akg > indiv_sum:
            print("  → Synergistic (combination > sum of parts)")
        else:
            print("  → Partially redundant (combination < sum of parts)")

STAGE 6: SYSTEMATIC KNOCKOUT SCREEN
Screening 33 candidates added to Tier 2 (100% O₂, cap = 21.97 mmol/gDW/hr)...
Tier 2 baseline: growth=0.9551, akg=3.0099

=== Top beneficial knockouts (added to Tier 2, 100% O₂) ===
added_knockout                                   name  growth  max_akg  improvement  growth_cost
         SUCDi Succinate dehydrogenase (irreversible)  0.9335   3.0868       0.0769       0.0215
           FUM                               Fumarase  0.8959   3.0868       0.0769       0.0591
           MDH                   Malate dehydrogenase  0.9401   3.0764       0.0665       0.0149
         GLUDy         Glutamate dehydrogenase (NADP)  0.9212   3.0392       0.0293       0.0338
           ICL                       Isocitrate lyase  0.9443   3.0275       0.0176       0.0107
          MALS                        Malate synthase  0.9443   3.0275       0.0176       0.0107

=== LETHAL knockouts (avoid) ===
added_knockout                                 name
         TYRTA   

**Interpreting the Results**

The screen was run at 100% O₂ (O₂ cap = 21.97 mmol/gDW/hr). At this fully aerobic condition, the TCA cycle runs strongly and the Tier 2 background already captures much of the available α-KG production gain. The additional improvements are therefore modest in absolute terms. Under microaerobic conditions (50% O₂), the same knockouts produce much larger effects — this is why Stage 2 showed Tier 2.5 (which includes ΔGLUDy) reaching 6.67 mmol/gDW/hr at 50% O₂, compared to Tier 2's 6.50.

**1. SUCDi and FUM (tied at +0.077)**

Succinate dehydrogenase (SUCDi) and fumarase (FUM) are adjacent steps in the lower TCA cycle: succinate → fumarate → malate. In Tier 2, both AKGDH paralogs are already knocked out, so the conventional TCA cycle is severed at α-KG. The only way succinate enters the network is via the glyoxylate shunt (ICL produces succinate from isocitrate). Knocking out SUCDi or FUM prevents this succinate from cycling back to OAA through the lower TCA branch, which forces more isocitrate through IDH toward α-KG instead of through the shunt. Because they block successive steps on the same linear path, they produce identical α-KG improvements under FBA.

However, their growth costs differ substantially: SUCDi costs only 0.022 while FUM costs 0.059. This makes **SUCDi the clearly preferable single target** — same benefit, less than half the fitness penalty. FUM is a new hit not in the predefined tiers and warrants experimental follow-up, but SUCDi should be tested first.

**2. MDH (+0.067, growth cost 0.015)**

Malate dehydrogenase catalyzes the forward TCA reaction malate + NAD⁺ → OAA + NADH under aerobic conditions. Knocking it out renders the oxidative TCA incomplete at the malate → OAA step, reducing OAA availability for citrate synthase and thereby limiting the rate at which carbon can cycle back through the upper TCA to consume α-KG. The cell must rely entirely on PPC (PEP carboxylase) to replenish OAA. MDH has the lowest growth cost of any beneficial knockout (0.015) and is experimentally validated: Chiang et al. (2025) included Δ_mdh_ in their 44 g/L α-KG strain from glycerol. The mechanistic rationale applies equally to glucose.

**3. GLUDy (+0.029, growth cost 0.034)**

Glutamate dehydrogenase is the single largest direct consumer of α-KG in the WT pFBA solution (7.83 mmol/gDW/hr, Stage 1). When GLUDy is knocked out, the cell must use the GS-GOGAT route for glutamate synthesis. Crucially, **GS-GOGAT consumes the same amount of α-KG per glutamate** (net: α-KG + NH₄⁺ + NADPH + ATP → Glu). The improvement in α-KG secretion comes not from reduced α-KG consumption per se, but from the **additional ATP cost** of the GS route. This ATP expenditure slightly constrains the growth objective, which under the ≥80% growth constraint releases more carbon for α-KG secretion. The modest gain at 100% O₂ (+0.029) becomes much larger at 50% O₂ — this is exactly the Tier 2.5 upgrade validated in Stage 2 (from 6.50 to 6.67 mmol/gDW/hr).

**4. ICL and MALS (tied at +0.018, growth cost 0.011)**

Isocitrate lyase and malate synthase are the two enzymes of the glyoxylate shunt. Blocking either forces all isocitrate through IDH → α-KG rather than through the bypass. The improvement is modest because the glyoxylate shunt is likely glucose-repressed in vivo via the IclR transcriptional repressor (which represses the _aceBAK_ operon under glucose growth conditions). FBA does not model transcriptional regulation, so the in-silico improvement is an upper bound; the in-vivo gain may be negligible. This is tested explicitly in Stage 12.

**5. Lethal knockouts**

- **ICDHyr** (isocitrate dehydrogenase, NADP-dependent): Essential because it generates α-KG from isocitrate. Without it, no α-KG is produced and biomass synthesis fails.
- **ASPTA** (aspartate transaminase): Essential for aspartate and downstream amino acid biosynthesis.
- **TYRTA, ILETA, PHETA1, ACOTA, SDPTA**: Aminotransferases that use α-KG as the amino group acceptor. Each is essential for synthesizing specific amino acids or biosynthetic intermediates. These should never be included in a production strain.

**6. Combination test: SUCDi + FUM**

The combination yields an improvement of +0.154, which equals the sum of individual improvements (0.077 + 0.077). This is expected: SUCDi and FUM block successive steps on the same linear path, so their effects are strictly additive under FBA. Combinations involving knockouts on _different_ pathways (e.g., SUCDi + MDH, or MDH + GLUDy) are tested in Stage 7.

**Connection to other stages:**

- Stage 1 identified GLUDy as the largest α-KG consumer (7.83 mmol/gDW/hr) — this screen confirms that knocking it out helps, but less than expected because GS-GOGAT is an efficient substitute.
- Stage 2 defined Tier 2.5 = Tier 2 + ΔGLUDy, which this screen validates. The larger gain at 50% O₂ (Stage 2) compared to 100% O₂ (this screen) confirms that GLUDy's benefit is oxygen-dependent.
- Stage 7 will test pairwise combinations of the top hits from this screen.
- Stage 12 will test whether ICL/MALS knockouts matter after accounting for glucose repression of the _aceBAK_ operon.
- Stage 14 (Literature-Optimized Complete Strain) will include ΔMDH based on both this screen and the Chiang et al. (2025) experimental validation.

**ELI5 Summary**

This stage is like testing every valve in a pipe network to see which ones, when closed, push more water (α-KG) toward the drain (export). Because the Tier 2 strain is already well-engineered, most valves do nothing useful — closing them either doesn't change the water flow or floods the system (lethal knockout).

The two best valves are on the lower TCA "return loop" — the succinate-to-fumarate valve (SUCDi) and the fumarase valve (FUM). Closing either prevents carbon from cycling back to the starting point (OAA), which forces more material through the α-KG export dock. The malate-to-OAA valve (MDH) works similarly but at the next step in the loop. The glutamate valve (GLUDy) helps somewhat but costs more growth — it's like restricting a department that's a major α-KG consumer, except the department finds a workaround (GOGAT) that uses the same amount of α-KG but costs more energy (ATP), indirectly freeing up a bit more carbon for export.

The screen shows modest gains at full oxygen because the factory is already running at peak efficiency. The same valves become much more important at half-oxygen (50% O₂), where the factory's waste-processing capacity is limited — exactly the condition tested in earlier stages.

---

**References for this stage:**

- Chiang, C.-J. et al. (2025). Metabolic Engineering of _E. coli_ for Overproduction of Alpha-Ketoglutarate Using Crude Glycerol. _J. Agric. Food Chem._ 73, 18346–18352.
- Li, M. et al. (2006). Effect of _sucA_ or _sucC_ gene knockout on the metabolism in _E. coli_ based on gene expressions, enzyme activities, intracellular metabolite concentrations and metabolic fluxes by ¹³C-labeling experiments. _Biochem. Eng. J._ 30, 286–296.
- Noh, M.H. et al. (2017). Precise flux redistribution to glyoxylate cycle for 5-aminolevulinic acid production in _E. coli_. _Process Biochem._ 56, 135–141.
- van 't Hof, M. et al. (2022). iHM1533: A genome-scale metabolic model for _E. coli_ Nissle 1917. _BMC Bioinformatics_ 23, 454.

## Stage 7: Combinatorial Optimization (Pairs + Overexpression)

**High-Level Summary**

This stage tests all pairwise combinations of the top six single-knockout hits from Stage 6, identifies synergistic pairs, and tests overexpression of PPC and CS via flux forcing. The best combination at 100% O₂ is **Tier 2 + ΔGLUDy + ΔSUCDi** (growth = 0.8879, α-KG = 3.2472, a 7.9% improvement over Tier 2 alone). At 50% O₂ the same pair yields α-KG = 6.7405. Overexpression of PPC has no effect (zero baseline flux on glucose), while CS overexpression above 1× is counterproductive under microaerobic conditions due to NADH-driven redox imbalance.

**Justification**

Single-knockout screens can miss synergistic interactions where two knockouts together improve production more than the sum of their individual effects. Stage 6 identified six beneficial knockouts; this stage tests all 15 pairwise combinations to find non-additive interactions. For example, ΔGLUDy imposes an ATP drain via the GS-GOGAT alternative (see Stage 6 interpretation), which reduces the cell's maximum growth capacity. When combined with ΔSUCDi — which blocks the oxidative lower TCA cycle from recycling carbon back to OAA — the reduced growth ceiling leaves more carbon available for a network where the recycling path is already severed. This creates a synergistic effect that exceeds the sum of individual improvements.

Overexpression is modeled in FBA by forcing a minimum flux through the target reaction. This is a stoichiometric test: if the constrained optimum does not improve, the reaction is not stoichiometrically rate-limiting in this genetic background. This does not rule out kinetic limitation, enzyme regulation, or other factors that FBA cannot capture. For targets with zero baseline flux (like PPC in Tier 2 on glucose), multiplicative forcing (3×, 5×, 10× of baseline) is undefined. The code substitutes a floor of 0.1 mmol/gDW/hr and tests absolute flux levels instead. This is stated explicitly in the output.

**Important context — PPC on glucose vs glycerol:** PPC has zero flux in Tier 2 because on glucose, PEP is consumed by the PTS for glucose import (1 PEP per glucose). Forcing PPC flux diverts PEP away from PTS, which reduces glucose uptake or bypasses pyruvate kinase (Pyk) at an ATP cost. This is why Chiang et al. (2025) used glycerol — which enters via GlpF/GlpK without consuming PEP — enabling PPC overexpression as a viable strategy. On our glucose-based system, PPC overexpression is not useful; heterologous PYC (which uses pyruvate, not PEP) is the correct anaplerotic strategy (tested in Stage 13).

**Algorithm**

1. Take the top 6 single-knockout hits from Stage 6: **SUCDi, FUM, MDH, GLUDy, ICL, MALS**.
2. Test all C(6,2) = 15 pairwise combinations added to Tier 2 at 100% and 50% O₂.
3. For the best pair at 100% O₂, compute synergy: compare pair improvement to the sum of the two individual Stage 6 improvements.
4. For PPC and CS overexpression: a. Compute baseline pFBA flux in Tier 2 at 100% O₂. b. For PPC: baseline = 0, so use absolute flux floor of 0.1 mmol/gDW/hr. c. For CS: baseline ≈ 5.9, use multiplicative forcing. d. Test 1×, 3×, 5×, 10× at 50% O₂ (the condition where bottlenecks are most apparent).

In [16]:
"""
Stage 7: Combinatorial Optimization (Pairs + Overexpression)
=============================================================
Prerequisite: Stages 0 and 6 completed.

PURPOSE:
  Test whether Stage 6's single-knockout hits interact synergistically
  when combined in pairs. Also test whether overexpression of PPC or CS
  (modeled as flux forcing) improves α-KG production.

WHY PAIRS?
  Stage 6 tested one additional knockout at a time on Tier 2. But two
  knockouts can interact: ΔGLUDy + ΔSUCDi might help more than the
  sum of their individual effects, because GLUDy's ATP drain lowers
  the growth ceiling (freeing carbon) while SUCDi prevents that freed
  carbon from recycling through the lower TCA.

WHY OVEREXPRESSION?
  Forcing minimum flux through PPC or CS tests whether these reactions
  are stoichiometrically rate-limiting. This is an FBA-level proxy for
  overexpression — it shows whether the model optimum improves under
  the artificial flux floor. It does NOT prove kinetic rate-limitation.
"""

import warnings
from itertools import combinations
from cobra.flux_analysis import pfba

# Suppress solver infeasibility warnings from CS×10 crash
warnings.filterwarnings("ignore", category=UserWarning, module="cobra.util.solver")

print("=" * 70)
print("STAGE 7: COMBINATORIAL OPTIMIZATION")
print("=" * 70)

# ─── 7.0: Build candidate list from Stage 6 results ─────────────────
# Use the actual Stage 6 output, not a hardcoded list.
# Stage 6 identified 6 beneficial hits (improves=True):
#   SUCDi, FUM, MDH, GLUDy, ICL, MALS
# ABTA did NOT cross the improvement threshold in Stage 6 and is excluded.
TOP_SINGLES = ["SUCDi", "FUM", "MDH", "GLUDy", "ICL", "MALS"]
TIER2_KOS = ["AKGDH", "AKGDH2", "LDH_D", "PTAr"]

print(f"Top Stage 6 singles for pairwise testing: {TOP_SINGLES}")
print(f"Number of pairs to test: C({len(TOP_SINGLES)},2) = "
      f"{len(list(combinations(TOP_SINGLES, 2)))}")

# ─── Helper function ─────────────────────────────────────────────────
def quick_production(model, kos, o2_frac):
    """
    Maximize α-KG at ≥80% of THIS design's own growth for given knockouts.

    Returns dict with 'growth' (µ_max) and 'max_akg' (α-KG at 80% floor).
    Uses the same per-mutant growth constraint as Stage 6.
    """
    o2_cap = O2_BASELINE * o2_frac
    with model:
        # Set medium
        model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
        model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(o2_cap)
        model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
        model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0

        # Apply all knockouts (Tier 2 base + additional)
        for ko in kos:
            model.reactions.get_by_id(ko).lower_bound = 0.0
            model.reactions.get_by_id(ko).upper_bound = 0.0

        # Step 1: Maximum growth with these knockouts
        model.objective = BIOMASS_ID
        model.objective_direction = "max"
        g_sol = model.optimize()
        mu = g_sol.objective_value if g_sol.status == "optimal" else 0.0

        # Step 2: Maximum α-KG at ≥80% of this mutant's growth
        akg = 0.0
        if mu > 0.01:
            with model:
                model.reactions.get_by_id(BIOMASS_ID).lower_bound = mu * 0.80
                model.objective = AKG_EX_ID
                model.objective_direction = "max"
                a_sol = model.optimize()
                akg = (a_sol.fluxes[AKG_EX_ID]
                       if a_sol.status == "optimal" else 0.0)

        return {"growth": round(mu, 4), "max_akg": round(akg, 4)}

# ─── 7.1: Tier 2 baselines for comparison ───────────────────────────
print("\n--- Tier 2 baselines ---")
baselines = {}
for o2_frac in [1.0, 0.50]:
    r = quick_production(model, TIER2_KOS, o2_frac)
    baselines[int(o2_frac * 100)] = r
    print(f"  {int(o2_frac*100)}% O₂: growth={r['growth']}, akg={r['max_akg']}")

# ─── 7.2: Pairwise combinations ─────────────────────────────────────
# Test all C(6,2) = 15 pairs at both 100% and 50% O₂.
combo_rows = []
for ko1, ko2 in combinations(TOP_SINGLES, 2):
    for o2_frac in [1.0, 0.50]:
        extended = TIER2_KOS + [ko1, ko2]
        r = quick_production(model, extended, o2_frac)
        o2_pct = int(o2_frac * 100)
        combo_rows.append({
            "ko1": ko1, "ko2": ko2,
            "o2_pct": o2_pct,
            **r,
            "improvement": round(r["max_akg"] - baselines[o2_pct]["max_akg"], 4),
            "growth_cost": round(baselines[o2_pct]["growth"] - r["growth"], 4),
            "viable": r["growth"] > 0.01,
        })

combo_df = pd.DataFrame(combo_rows)
combo_df.to_csv(OUTPUT / "stage7_pairwise_combos.csv", index=False)

for o2 in [100, 50]:
    sub = combo_df[(combo_df["o2_pct"] == o2) & combo_df["viable"]]
    sub = sub.sort_values("max_akg", ascending=False)
    print(f"\n=== Top 5 pairs at {o2}% O₂ ===")
    print(sub.head(5)[["ko1", "ko2", "growth", "max_akg",
                        "improvement", "growth_cost"]].to_string(index=False))

# ─── 7.3: Synergy check for the best pair at 100% O₂ ────────────────
# Compare pair improvement to sum of individual Stage 6 improvements.
# Stage 6 individual improvements (from verified output):
STAGE6_IMPROVEMENTS = {
    "SUCDi": 0.0769, "FUM": 0.0769, "MDH": 0.0665,
    "GLUDy": 0.0293, "ICL": 0.0176, "MALS": 0.0176,
}

best_100 = (combo_df[(combo_df["o2_pct"] == 100) & combo_df["viable"]]
            .sort_values("max_akg", ascending=False)
            .head(1))

if not best_100.empty:
    ko1 = best_100.iloc[0]["ko1"]
    ko2 = best_100.iloc[0]["ko2"]
    pair_imp = best_100.iloc[0]["improvement"]
    indiv_sum = STAGE6_IMPROVEMENTS.get(ko1, 0) + STAGE6_IMPROVEMENTS.get(ko2, 0)

    print(f"\n=== Synergy check: {ko1} + {ko2} at 100% O₂ ===")
    print(f"  Individual improvements: "
          f"{STAGE6_IMPROVEMENTS.get(ko1,0):.4f} + "
          f"{STAGE6_IMPROVEMENTS.get(ko2,0):.4f} = {indiv_sum:.4f}")
    print(f"  Pair improvement:        {pair_imp:.4f}")
    if pair_imp > indiv_sum * 1.1:
        print(f"  → SYNERGISTIC ({pair_imp/indiv_sum:.1f}× the additive expectation)")
    elif pair_imp > indiv_sum * 0.9:
        print("  → Approximately additive")
    else:
        print("  → Partially redundant")

# ─── 7.4: PPC and CS overexpression (flux forcing) ──────────────────
# Test at 50% O₂ where bottlenecks are most apparent (Stage 3/5).
print("\n--- Overexpression flux-forcing (50% O₂, Tier 2 background) ---")

for target in ["PPC", "CS"]:
    # Get baseline flux via pFBA in Tier 2 at 100% O₂
    with model:
        model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
        model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(O2_BASELINE)
        for ko in TIER2_KOS:
            model.reactions.get_by_id(ko).lower_bound = 0.0
            model.reactions.get_by_id(ko).upper_bound = 0.0
        model.objective = BIOMASS_ID
        sol = pfba(model)
        raw_flux = sol.fluxes.get(target, 0)

    print(f"\n  {target} baseline flux in Tier 2 (pFBA, 100% O₂): {raw_flux:.4f}")

    # Handle zero-baseline case (PPC on glucose)
    if abs(raw_flux) < 1e-6:
        print(f"  NOTE: {target} baseline is ~0. Multiplicative forcing is undefined.")
        print(f"  Using absolute flux floor of 0.1 mmol/gDW/hr instead.")
        base_flux = 0.1
    else:
        base_flux = abs(raw_flux)

    # Test forcing at 1×, 3×, 5×, 10× of base_flux under 50% O₂
    for mult in [1, 3, 5, 10]:
        forced = base_flux * mult
        with model:
            model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
            model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(O2_BASELINE * 0.5)
            model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
            model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0
            for ko in TIER2_KOS:
                model.reactions.get_by_id(ko).lower_bound = 0.0
                model.reactions.get_by_id(ko).upper_bound = 0.0

            # Force minimum flux through target reaction
            model.reactions.get_by_id(target).lower_bound = forced

            # Maximize growth under the forced flux
            model.objective = BIOMASS_ID
            model.objective_direction = "max"
            g_sol = model.optimize()
            mu = g_sol.objective_value if g_sol.status == "optimal" else 0.0

            # If viable, maximize α-KG at ≥80% growth
            akg = 0.0
            if mu > 0.01:
                with model:
                    model.reactions.get_by_id(target).lower_bound = forced
                    model.reactions.get_by_id(BIOMASS_ID).lower_bound = mu * 0.80
                    model.objective = AKG_EX_ID
                    model.objective_direction = "max"
                    a_sol = model.optimize()
                    akg = (a_sol.fluxes[AKG_EX_ID]
                           if a_sol.status == "optimal" else 0.0)

            status = "" if mu > 0.01 else " ← INFEASIBLE"
            print(f"    {target}×{mult:<2} (forced={forced:.2f}, 50% O₂): "
                  f"growth={mu:.4f}, akg={akg:.4f}{status}")

STAGE 7: COMBINATORIAL OPTIMIZATION
Top Stage 6 singles for pairwise testing: ['SUCDi', 'FUM', 'MDH', 'GLUDy', 'ICL', 'MALS']
Number of pairs to test: C(6,2) = 15

--- Tier 2 baselines ---
  100% O₂: growth=0.9551, akg=3.0099
  50% O₂: growth=0.6084, akg=6.5046

=== Top 5 pairs at 100% O₂ ===
  ko1   ko2  growth  max_akg  improvement  growth_cost
SUCDi GLUDy  0.8879   3.2472       0.2373       0.0672
  MDH GLUDy  0.9009   3.1450       0.1351       0.0542
SUCDi   FUM  0.8959   3.0868       0.0769       0.0592
SUCDi   MDH  0.9335   3.0868       0.0769       0.0216
SUCDi   ICL  0.9335   3.0868       0.0769       0.0216

=== Top 5 pairs at 50% O₂ ===
  ko1   ko2  growth  max_akg  improvement  growth_cost
SUCDi GLUDy  0.5532   6.7405       0.2359       0.0552
GLUDy   ICL  0.5532   6.7405       0.2359       0.0552
GLUDy  MALS  0.5532   6.7405       0.2359       0.0552
  MDH GLUDy  0.5557   6.6810       0.1764       0.0527
  FUM GLUDy  0.5496   6.6559       0.1513       0.0588

=== Synergy ch

**Interpreting the Results**

**Pairwise knockout combinations:**

The best pair at 100% O₂ is **GLUDy + SUCDi** (α-KG = 3.2472, improvement = +0.237 over Tier 2 baseline of 3.0099). This represents a **synergistic** interaction: the individual Stage 6 improvements sum to only 0.106 (GLUDy: 0.029 + SUCDi: 0.077), but the combination achieves 0.237 — more than double the additive expectation. The mechanism is that ΔGLUDy imposes an ATP drain (via the more expensive GS-GOGAT route for glutamate synthesis), which lowers µ_max. The lower growth ceiling means 80% of µ_max is also lower, freeing more carbon. ΔSUCDi then prevents this freed carbon from cycling through the lower TCA back to OAA, forcing it toward α-KG secretion instead.

At 50% O₂, several GLUDy-containing pairs tie at 6.7405 (GLUDy+SUCDi, GLUDy+ICL, GLUDy+MALS, GLUDy+FUM). The differences between these pairs are flattened because the microaerobic condition already provides the dominant production-driving constraint (NADH imbalance, Stage 5). The second knockout matters less when oxygen limitation is already forcing most of the flux redistribution.

**PPC overexpression — no effect on glucose:**

PPC carries zero flux in the Tier 2 pFBA solution. Even forcing up to 1.0 mmol/gDW/hr produces no improvement (growth and α-KG remain essentially unchanged). This is because PPC uses PEP as substrate, and on glucose, PEP is consumed by the PTS for glucose import. Forcing PPC flux either diverts PEP from PTS (reducing glucose uptake) or bypasses pyruvate kinase (losing ATP). The model's constrained optimum finds no benefit.

This result is specific to glucose. Chiang et al. (2025) successfully overexpressed _ppc_ because they used glycerol, which imports via GlpF/GlpK without consuming PEP. On glycerol, PEP is freely available for PPC. This glucose-vs-glycerol distinction is critical for translating published α-KG engineering strategies to our system (see also Stage 13, which tests heterologous PYC as the correct anaplerotic strategy for glucose).

**CS overexpression — counterproductive under microaerobic conditions:**

Forcing CS at its aerobic baseline rate (5.9 mmol/gDW/hr) under 50% O₂ marginally improves α-KG to 6.71 — a modest gain from maintaining full TCA entry flux despite reduced oxygen. However, forcing CS to 3× (17.7) crashes growth by 23% and α-KG by 44%. At 5×, growth halves. At 10×, the model is infeasible.

The mechanism is a **NADH redox crisis** (see Stage 5): CS initiates the oxidative TCA cycle, which generates NADH at IDH and other steps. At 50% O₂, the ETC is already running at half capacity and cannot reoxidize the additional NADH. With lactate and acetate routes blocked in Tier 2, there is no way to dispose of the excess NADH. The solver correctly returns infeasibility.

The practical lesson: do **not** overexpress CS under microaerobic conditions without also providing an NADH sink (e.g., NADH oxidase from _L. lactis_). Stage 10 tests an alternative approach — replacing native CS with feedback-insensitive CggltA, which maintains the same baseline flux without amplifying it.

**Connection to other stages:**

- Stage 6 provided the single-knockout inputs; this stage tests their pairwise interactions.
- Stage 5 (NADH Redox Balance) explains why CS overexpression fails at 50% O₂.
- Stage 10 (CS Feedback) tests the complementary strategy: replacing native CS with insensitive CggltA rather than overexpressing it.
- Stage 13 (PYC vs PPC) confirms that PYC (using pyruvate) is the correct anaplerotic strategy on glucose, not PPC (using PEP).
- Stage 14 (LitOpt) combines the best pair (GLUDy + additional knockouts) with PYC and ΔMDH for the final strain.

**ELI5 Summary**

Stage 6 tested closing one valve at a time. Stage 7 tests closing two valves simultaneously. The GLUDy + SUCDi pair is synergistic — like sealing both a leak in the ceiling (GLUDy drawing off α-KG for glutamate) and a drain in the floor (SUCDi recycling carbon through the lower TCA). Either fix alone helps a little; both together help a lot because the water (carbon) has nowhere to go except the α-KG export dock.

For the "turbocharge" strategy (overexpression): forcing more PPC on glucose doesn't work because the factory needs PEP to pay the entry fee for glucose. It's like trying to power a backup generator with the fuel that runs the main gate — there's nothing left over. Forcing more CS at half-oxygen is even worse: it generates more "used batteries" (NADH) than the half-powered recharger (ETC) can handle, causing a factory-wide power outage.

---

**References for this stage:**

- Chiang, C.-J. et al. (2025). Metabolic Engineering of _E. coli_ for Overproduction of Alpha-Ketoglutarate Using Crude Glycerol. _J. Agric. Food Chem._ 73, 18346–18352.
- Li, M. et al. (2006). Effect of _sucA_ or _sucC_ gene knockout on the metabolism in _E. coli_ based on gene expressions, enzyme activities, intracellular metabolite concentrations and metabolic fluxes by ¹³C-labeling experiments. _Biochem. Eng. J._ 30, 286–296.
- Noh, M.H. et al. (2017). Production of 5-aminolevulinic acid from succinyl-CoA in _E. coli_. _Process Biochem._ 56, 135–141.
- van 't Hof, M. et al. (2022). iHM1533: A genome-scale metabolic model for _E. coli_ Nissle 1917. _BMC Bioinformatics_ 23, 454.



## Stage 8: Transport & Export Analysis (KgtP)

**High-Level Summary**

This stage maps every reaction that moves α-KG across compartment boundaries in iHM1533, identifies KgtP (AKGt2rpp) as the sole inner-membrane α-KG transporter, and demonstrates that **all α-KG "secretion" in every FBA solution occurs through AKGt2rpp running in reverse** — a thermodynamically disfavored direction for an H⁺:α-KG symporter operating against the proton motive force. When AKGt2rpp is correctly constrained to import-only (lb = 0), modeled α-KG secretion drops to exactly zero in both the Tier 2 and Stage 7 best-pair backgrounds.

This is a **model completeness finding**, not an absolute biological verdict. The iHM1533 model lacks a dedicated α-KG export reaction, so the FBA solver exploits the only available route (reverse KgtP). Published _E. coli_ metabolic engineering studies achieve substantial α-KG titers without heterologous exporters, implying that passive efflux mechanisms (outer membrane porin diffusion at high intracellular concentrations, non-specific MFS carriers) exist biologically but are unrepresented in the model. For the _C. elegans_ delivery context specifically, pharyngeal grinding destroys most ingested bacteria, releasing intracellular contents directly.

An export capacity sweep quantifies the minimum transporter throughput needed before the bottleneck shifts from export to the intracellular pathway — this number informs heterologous exporter selection.

**Justification**

KgtP was identified by Seol & Shatkin (1991) as the _E. coli_ α-KG transporter gene. In a follow-up study, Seol & Shatkin (1992) characterized it as an H⁺ symporter with a Km of approximately 13 µM for α-KG. The proton motive force (~−180 mV in aerobic _E. coli_) strongly favors inward H⁺ movement, making the reverse direction (export, requiring outward H⁺ movement) thermodynamically disfavored under physiological conditions.

**A note on KgtP regulation:** The Seol & Shatkin (1992) paper described kgtP as "constitutively expressed." Subsequent work has shown that kgtP expression is more nuanced: the gene lies within the σˢ-regulated _csiD–ygaF–gabDTP_ region and is upregulated under carbon limitation and stationary phase. During exponential growth on glucose — the condition modeled here — kgtP expression is at basal levels. For the FBA model, this regulation is irrelevant (FBA does not model gene expression), but for the wet lab it means KgtP protein levels during exponential growth may be lower than during stationary phase.

The iHM1533 model (van 't Hof et al., 2022) represents KgtP as AKGt2rpp with bounds [−1000, 1000], allowing free reversibility. This is standard practice in genome-scale models, which do not encode thermodynamic directionality constraints. All α-KG appearing at EX_akg_e in our production solutions transits through AKGt2rpp at negative flux (the export direction). AKGtex (periplasm ↔ extracellular) is a porin-mediated diffusion step (OR-GPR: CIW80_19480 or CIW80_22660 or CIW80_25725 or CIW80_05020) and is thermodynamically bidirectional — once α-KG reaches the periplasm it can freely equilibrate with the extracellular space.

**Why does real _E. coli_ secrete α-KG without a heterologous exporter?** Published studies achieve α-KG titers of 5–70 g/L from engineered _E. coli_ strains without exotic export machinery (Li et al., 2006). At the intracellular α-KG concentrations reached in engineered strains (potentially tens of millimolar), passive outward diffusion through outer membrane porins (OmpF, OmpC) and overflow efflux through broad-specificity MFS transporters likely account for the observed export. These mechanisms are not encoded in iHM1533. The stage's finding should therefore be read as: the model predicts zero production when the known thermodynamic directionality of KgtP is enforced — indicating a model gap, not that biological export is impossible.

**Algorithm**

1. Identify all reactions involving any α-KG metabolite that cross compartment boundaries.
2. Verify KgtP = AKGt2rpp and its GPR (CIW80_06765).
3. Verify AKGt2rpp flux direction in Tier 2 production solutions at 100%, 50%, and 20% O₂.
4. Repeat in the Stage 7 best-pair background (Tier 2 + ΔGLUDy + ΔSUCDi) to characterize transport in the highest-producing genetic context.
5. Test AKGt2rpp import-only (lb = 0, ub = 1000) in both backgrounds: does blocking the export direction eliminate modeled secretion?
6. Test full AKGt2rpp knockout (lb = ub = 0) in both backgrounds: does blocking all KgtP flux affect growth?
7. Export capacity sweep: constrain EX_akg_e upper bound from 0 to unconstrained at 50% O₂ and identify the minimum export capacity at which production becomes pathway-limited rather than export-limited.

In [18]:
"""
Stage 8: Transport & Export Analysis (KgtP)
=============================================
Prerequisite: Stages 0 and 7 completed.

PURPOSE:
  Map the α-KG transport network in iHM1533. Identify that the model's
  only cytoplasm→periplasm route (AKGt2rpp / KgtP) runs in reverse in
  all production solutions — a thermodynamically disfavored direction
  for an H⁺ symporter. Quantify what happens when the correct import-
  only constraint is applied. Sweep export capacity to find the minimum
  transporter throughput needed for a heterologous exporter.

KEY BIOLOGICAL CONTEXT:
  KgtP was characterized as an α-KG uptake permease (Seol & Shatkin,
  1991, PNAS 88:3802). The proton motive force (~-180 mV) favors
  inward H⁺ movement, making outward (export) flux thermodynamically
  disfavored. However, the iHM1533 model encodes AKGt2rpp as freely
  reversible [-1000, 1000] — a standard GEM simplification.

  Published E. coli α-KG production studies achieve titers without
  heterologous exporters (Li et al., 2006), implying that biological
  efflux mechanisms (porin diffusion, overflow) exist but are absent
  from the model. This stage's finding is a MODEL-LEVEL result.
"""

import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="cobra.util.solver")

print("=" * 70)
print("STAGE 8: TRANSPORT & EXPORT ANALYSIS")
print("=" * 70)

TIER2_KOS  = ["AKGDH", "AKGDH2", "LDH_D", "PTAr"]
STAGE7_KOS = TIER2_KOS + ["GLUDy", "SUCDi"]  # Best Stage 7 pair
KGTP_ID    = "AKGt2rpp"

# ─── 8.1: Map all α-KG transport/exchange reactions ─────────────────
# Scan every reaction in the model for metabolites containing "akg".
# Keep only reactions that either (a) span multiple compartments
# (transport) or (b) have a single metabolite (exchange/sink).
# This exhaustively identifies all routes α-KG can take in/out of
# the cytoplasm.
print("\n--- All α-KG transport/exchange reactions ---")
transport_rows = []
for rxn in model.reactions:
    akg_mets = [m for m in rxn.metabolites if "akg" in m.id.lower()]
    if not akg_mets:
        continue
    compartments = set(m.compartment for m in rxn.metabolites)
    if len(compartments) > 1 or len(rxn.metabolites) == 1:
        print(f"  {rxn.id:15s} | {rxn.name[:55]:55s}")
        print(f"  {'':15s} | {rxn.reaction}")
        print(f"  {'':15s} | GPR: {rxn.gene_reaction_rule}")
        print(f"  {'':15s} | bounds=[{rxn.lower_bound}, {rxn.upper_bound}]")
        transport_rows.append({
            "id": rxn.id, "name": rxn.name,
            "reaction": rxn.reaction, "gpr": rxn.gene_reaction_rule,
            "lb": rxn.lower_bound, "ub": rxn.upper_bound,
        })
pd.DataFrame(transport_rows).to_csv(OUTPUT / "stage8_akg_transport.csv", index=False)

# ─── 8.2: Verify KgtP flux direction in production solutions ────────
# For each genetic background × O₂ level, solve the two-step production
# problem (max growth → max α-KG at ≥80% growth) and record the KgtP
# flux. If KgtP is negative, the solver is running it in reverse (export).
#
# REACTION CONVENTION in iHM1533:
#   AKGt2rpp:  akg_p + h_p ⇌ akg_c + h_c
#   Positive flux = IMPORT (periplasm → cytoplasm, favored by PMF)
#   Negative flux = EXPORT (cytoplasm → periplasm, against PMF)
print("\n--- KgtP flux direction in production solutions ---")
flux_rows = []
for label, kos in [("Tier 2", TIER2_KOS),
                    ("Stage 7 best (T2+ΔGLUDy+ΔSUCDi)", STAGE7_KOS)]:
    print(f"\n  Background: {label}")
    for o2_frac in [1.0, 0.50, 0.20]:
        with model:
            # Set medium and knockouts
            model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
            model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(O2_BASELINE * o2_frac)
            model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
            model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0
            for ko in kos:
                model.reactions.get_by_id(ko).lower_bound = 0.0
                model.reactions.get_by_id(ko).upper_bound = 0.0

            # Step 1: Maximum growth for this background + O₂
            model.objective = BIOMASS_ID
            model.objective_direction = "max"
            g_sol = model.optimize()
            mu = g_sol.objective_value if g_sol.status == "optimal" else 0.0

            # Step 2: Maximum α-KG at ≥80% of that growth
            akg = kgtp_flux = akgtex_flux = 0.0
            if mu > 0.01:
                with model:
                    model.reactions.get_by_id(BIOMASS_ID).lower_bound = mu * 0.80
                    model.objective = AKG_EX_ID
                    model.objective_direction = "max"
                    a_sol = model.optimize()
                    if a_sol.status == "optimal":
                        kgtp_flux   = a_sol.fluxes[KGTP_ID]
                        akgtex_flux = a_sol.fluxes["AKGtex"]
                        akg         = a_sol.fluxes[AKG_EX_ID]

            direction = "EXPORT (reverse, ↑PMF)" if kgtp_flux < -0.01 else "import/zero"
            print(f"    {int(o2_frac*100):3d}% O₂: "
                  f"KgtP={kgtp_flux:+.4f} ({direction}), "
                  f"AKGtex={akgtex_flux:+.4f}, "
                  f"EX_akg_e={akg:.4f}")
            flux_rows.append({
                "background": label, "o2_pct": int(o2_frac * 100),
                "kgtp_flux": round(kgtp_flux, 4),
                "akgtex_flux": round(akgtex_flux, 4),
                "akg_secretion": round(akg, 4),
            })

pd.DataFrame(flux_rows).to_csv(OUTPUT / "stage8_flux_directions.csv", index=False)

# ─── 8.3: Import-only constraint — does model secretion survive? ─────
# Block the thermodynamically disfavored export direction (negative flux)
# while preserving the native uptake function (positive flux).
# lb=0 prevents reverse (export); ub=1000 allows forward (import).
print("\n--- AKGt2rpp import-only constraint (lb=0) ---")
import_only_rows = []
for label, kos in [("Tier 2", TIER2_KOS),
                    ("Stage 7 best", STAGE7_KOS)]:
    print(f"\n  Background: {label}")
    for o2_frac in [1.0, 0.50, 0.20]:
        with model:
            model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
            model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(O2_BASELINE * o2_frac)
            model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
            model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0
            for ko in kos:
                model.reactions.get_by_id(ko).lower_bound = 0.0
                model.reactions.get_by_id(ko).upper_bound = 0.0
            # Import-only: lb=0 blocks export; ub=1000 allows import
            model.reactions.get_by_id(KGTP_ID).lower_bound = 0.0
            model.reactions.get_by_id(KGTP_ID).upper_bound = 1000.0

            model.objective = BIOMASS_ID
            model.objective_direction = "max"
            g_sol = model.optimize()
            mu = g_sol.objective_value if g_sol.status == "optimal" else 0.0

            akg = 0.0
            if mu > 0.01:
                with model:
                    # Only new constraint vs outer: biomass floor + objective switch
                    model.reactions.get_by_id(BIOMASS_ID).lower_bound = mu * 0.80
                    model.objective = AKG_EX_ID
                    model.objective_direction = "max"
                    a_sol = model.optimize()
                    if a_sol.status == "optimal":
                        akg = a_sol.fluxes[AKG_EX_ID]

            print(f"    {int(o2_frac*100):3d}% O₂: growth={mu:.4f}, α-KG={akg:.4f}")
            import_only_rows.append({
                "background": label, "o2_pct": int(o2_frac * 100),
                "growth": round(mu, 4), "akg": round(akg, 4),
            })

pd.DataFrame(import_only_rows).to_csv(OUTPUT / "stage8_import_only.csv", index=False)

# ─── 8.4: Full KgtP knockout (lb = ub = 0) ──────────────────────────
# Does blocking ALL KgtP flux (import AND export) affect growth?
# This distinguishes between "the cell needs KgtP for import" vs
# "KgtP is dispensable." If growth is unchanged, KgtP is not needed
# for importing α-KG under these medium conditions (no exogenous α-KG).
print("\n--- Full KgtP knockout (lb = ub = 0) ---")
for label, kos in [("Tier 2", TIER2_KOS),
                    ("Stage 7 best", STAGE7_KOS)]:
    print(f"\n  Background: {label}")
    for o2_frac in [1.0, 0.50]:
        with model:
            model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
            model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(O2_BASELINE * o2_frac)
            model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
            model.reactions.get_by_id(AKG_EX_ID).upper_bound = 1000.0
            for ko in kos:
                model.reactions.get_by_id(ko).lower_bound = 0.0
                model.reactions.get_by_id(ko).upper_bound = 0.0
            # Full knockout: no flux in either direction
            model.reactions.get_by_id(KGTP_ID).lower_bound = 0.0
            model.reactions.get_by_id(KGTP_ID).upper_bound = 0.0

            model.objective = BIOMASS_ID
            g_sol = model.optimize()
            mu = g_sol.objective_value if g_sol.status == "optimal" else 0.0
            print(f"    {int(o2_frac*100):3d}% O₂: growth={mu:.4f} "
                  f"(unchanged = KgtP import not needed on this medium)")

print("""
MODEL FINDING (not a biological verdict):
  When AKGt2rpp is constrained to import-only (lb=0), the model predicts
  zero α-KG secretion in ALL backgrounds and oxygen levels. Growth is
  completely unaffected — the cell does not need to export α-KG to grow.

  Interpretation:
  1. In iHM1533, the sole cytoplasm→periplasm route for α-KG is reverse
     AKGt2rpp. The model has no dedicated export reaction.
  2. Reverse AKGt2rpp is thermodynamically disfavored (requires H⁺
     movement against the PMF, ~-180 mV).
  3. Published E. coli fermentation studies achieve α-KG titers of
     5-70 g/L WITHOUT heterologous exporters (Li et al. 2006),
     indicating that biological export mechanisms exist that are absent
     from the model (OmpF/OmpC porin diffusion at high intracellular
     concentration; non-specific MFS carriers; overflow efflux).
  4. For the C. elegans delivery context, pharyngeal grinding lyses
     most ingested bacteria, releasing intracellular α-KG regardless
     of active export.
  5. Engineering a validated heterologous exporter remains the best
     strategy for maximizing secreted titers; see Step 8.5 for the
     required minimum capacity.

  References: Seol & Shatkin 1991 PNAS; Seol & Shatkin 1992 JBC.
""")

# ─── 8.5: Export capacity sweep ──────────────────────────────────────
# Constrain EX_akg_e upper bound across a range while leaving KgtP
# unconstrained (the model's default). This answers: "What minimum
# transporter throughput is needed before the bottleneck shifts from
# export to pathway flux?" The transition from export-limited to
# pathway-limited gives the design target for a heterologous exporter.
#
# Run at 50% O₂ (most production-relevant condition from Stages 3/7).
print("\n--- Export capacity sweep (EX_akg_e upper bound, 50% O₂) ---")

sweep_caps = [0, 0.5, 1, 2, 3, 4, 5, 6, 7, 8, 10, 1000]
sweep_rows = []

for label, kos in [("Tier 2", TIER2_KOS),
                    ("Stage 7 best", STAGE7_KOS)]:
    print(f"\n  Background: {label}")
    print(f"  {'cap':>6s}  {'growth':>8s}  {'α-KG':>8s}  {'status':>20s}")
    for cap in sweep_caps:
        with model:
            model.reactions.get_by_id(GLC_EX_ID).lower_bound = -10.0
            model.reactions.get_by_id(O2_EX_ID).lower_bound = -abs(O2_BASELINE * 0.50)
            model.reactions.get_by_id(AKG_EX_ID).lower_bound = 0.0
            model.reactions.get_by_id(AKG_EX_ID).upper_bound = float(cap)
            for ko in kos:
                model.reactions.get_by_id(ko).lower_bound = 0.0
                model.reactions.get_by_id(ko).upper_bound = 0.0

            model.objective = BIOMASS_ID
            model.objective_direction = "max"
            g_sol = model.optimize()
            mu = g_sol.objective_value if g_sol.status == "optimal" else 0.0

            akg = 0.0
            if mu > 0.01:
                with model:
                    model.reactions.get_by_id(BIOMASS_ID).lower_bound = mu * 0.80
                    model.objective = AKG_EX_ID
                    model.objective_direction = "max"
                    a_sol = model.optimize()
                    if a_sol.status == "optimal":
                        akg = a_sol.fluxes[AKG_EX_ID]

            # Export-limited if α-KG ≈ cap (production hits the ceiling)
            if cap < 999 and abs(akg - cap) < 0.05:
                status = "export-limited"
            else:
                status = "pathway-limited"

            print(f"  {cap:6.1f}  {mu:8.4f}  {akg:8.4f}  {status:>20s}")
            sweep_rows.append({
                "background": label, "export_cap": cap,
                "growth": round(mu, 4), "akg": round(akg, 4),
                "status": status,
            })

pd.DataFrame(sweep_rows).to_csv(OUTPUT / "stage8_export_sweep.csv", index=False)

print("""
  EXPORT SWEEP INTERPRETATION:
  Rows marked 'export-limited': increasing transporter capacity would
  improve predicted titers. The transition to 'pathway-limited' gives
  the MINIMUM export capacity needed to stop being transporter-bottlenecked.
  Any heterologous exporter must sustain at least this flux at relevant
  intracellular α-KG concentrations.
""")

STAGE 8: TRANSPORT & EXPORT ANALYSIS

--- All α-KG transport/exchange reactions ---
  EX_akg_e        | 2-Oxoglutarate exchange                                
                  | akg_e --> 
                  | GPR: 
                  | bounds=[0.0, 1000.0]
  AKGt2rpp        | 2-oxoglutarate reversible transport via symport (peripl
                  | akg_p + h_p <=> akg_c + h_c
                  | GPR: CIW80_06765
                  | bounds=[-1000.0, 1000.0]
  AKGtex          | Alpha-ketoglutarate transport via diffusion (extracellu
                  | akg_e <=> akg_p
                  | GPR: CIW80_19480 or CIW80_22660 or CIW80_25725 or CIW80_05020
                  | bounds=[-1000.0, 1000.0]

--- KgtP flux direction in production solutions ---

  Background: Tier 2
    100% O₂: KgtP=-3.0099 (EXPORT (reverse, ↑PMF)), AKGtex=-3.0099, EX_akg_e=3.0099
     50% O₂: KgtP=-6.5046 (EXPORT (reverse, ↑PMF)), AKGtex=-6.5046, EX_akg_e=6.5046
     20% O₂: KgtP=-3.3786 (EXPORT (reverse, ↑PMF)), AK

**Interpreting the Results**

**AKGt2rpp flux direction:** In every production solution — across both genetic backgrounds and all oxygen levels — AKGt2rpp carries negative flux exactly equal in magnitude to the EX_akg_e flux. For Tier 2: KgtP = −3.0099 at 100% O₂, −6.5046 at 50% O₂, −3.3786 at 20% O₂. In the Stage 7 best-pair background (Tier 2 + ΔGLUDy + ΔSUCDi): KgtP = −3.2472, −6.7405, −3.4901 at the same oxygen levels. AKGtex carries equal positive flux in every case — confirming the full export chain is: reverse AKGt2rpp (cytoplasm → periplasm) → AKGtex (periplasm → extracellular) → EX_akg_e (system boundary).

**Import-only constraint:** Applying lb = 0 to AKGt2rpp eliminates all modeled α-KG secretion in both backgrounds at all oxygen levels. Growth is completely unaffected — the cell does not need to export α-KG to grow; it only "exported" in previous stages because we forced the solver to maximize the secretion objective, and reverse KgtP was the only coded route available.

**Full KgtP knockout:** Growth is identical to the import-only case. On glucose minimal medium with no exogenous α-KG, KgtP's import function is not utilized. This confirms that deleting _kgtP_ in the wet lab would have no growth consequence under these conditions — useful information if genetic simplification of the strain is desired.

**What this means biologically — the critical nuance:** This is a model completeness issue. Published _E. coli_ metabolic engineering studies achieve substantial α-KG titers in fermentation broth without any heterologous exporter (Li et al., 2006). At the high intracellular α-KG concentrations reached in engineered strains, passive outward diffusion through outer membrane porins OmpF and OmpC (the dominant non-selective channels for small organic anions), as well as potential overflow efflux through broad-specificity MFS transporters, likely account for the observed export. These mechanisms are not encoded in iHM1533. For the _C. elegans_ plate context specifically, pharyngeal grinding destroys most ingested bacteria, releasing intracellular contents directly into the intestinal lumen regardless of active export capacity.

**Export capacity sweep — minimum transporter requirement:** The sweep shows that production is export-limited for any EX_akg_e cap below ~6.5 mmol/gDW/hr (Tier 2) or ~6.7 mmol/gDW/hr (Stage 7 best). Below these thresholds, predicted α-KG secretion equals the cap exactly — the exit is the bottleneck, not the pathway. At caps ≥ 7 mmol/gDW/hr, production saturates at the pathway-imposed ceiling (6.5046 for Tier 2, 6.7405 for Stage 7 best). The practical implication: any heterologous exporter must sustain approximately 6.5–6.8 mmol/gDW/hr at 50% O₂ to avoid being the rate-limiting step. Note that growth is completely unaffected across the sweep — the export cap constrains only the production objective, not biomass.

**Connection to other stages:**

- Stage 7 identified the best genetic background (Tier 2 + ΔGLUDy + ΔSUCDi); this stage characterizes transport in that background.
- Stage 5 (NADH Redox Balance) explains the oxygen-dependence of the production ceiling that the export sweep saturates toward.
- Stage 9 (Heterologous Exporter Modeling) tests what happens when a synthetic export reaction is added to the model with KgtP constrained to import-only, including energy-cost variants (PMF-dissipating, ATP-consuming).
- Stage 13 (PYC vs PPC) addresses the anaplerotic limitation on total α-KG flux, which sets the pathway ceiling the export sweep reaches.

**ELI5 Summary**

The factory's only loading dock (KgtP) was designed as an _unloading_ dock — it brings raw materials in from outside, powered by the factory's electric potential (proton motive force). In the computer model, the dock is marked bidirectional (a simplification), so the optimizer cheerfully runs trucks backwards out through the entrance. When we correct the model to only allow inward trucks, the model predicts zero outbound shipments.

But here's the twist: real factories have been shown to ship product out anyway, at rates the model can't explain. The likely explanation is that the walls (outer membrane porins) are leaky enough that when product piles up high enough inside, it seeps through the cracks. The model doesn't know about the leaky walls. And if the customer (the worm) just demolishes the factory to get the goods (pharyngeal grinding), the loading dock design doesn't matter at all.

The capacity sweep tells us: if we do build a dedicated outbound dock (heterologous exporter), it needs to handle roughly 6.5–6.8 mmol per gram of cells per hour. Below that, the dock itself is the bottleneck; above that, the internal production pathway is what limits output.

---

**References for this stage:**

- Seol, W. & Shatkin, A.J. (1991). _Escherichia coli_ kgtP encodes an alpha-ketoglutarate transporter. _Proc. Natl. Acad. Sci. USA_ **88**, 3802–3806.
- Seol, W. & Shatkin, A.J. (1992). _Escherichia coli_ alpha-ketoglutarate permease is a constitutively expressed proton symporter. _J. Biol. Chem._ **267**, 6409–6413.
- Li, M. et al. (2006). Effect of _sucA_ or _sucC_ gene knockout on the metabolism in _E. coli_ based on gene expressions, enzyme activities, intracellular metabolite concentrations and metabolic fluxes by ¹³C-labeling experiments. _Biochem. Eng. J._ **30**, 286–296.
- van 't Hof, M. et al. (2022). iHM1533: A genome-scale metabolic model for _E. coli_ Nissle 1917. _BMC Bioinformatics_ **23**, 454.
